In [ ]:
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],  # Ubuntu 推荐
    "font.size": 20,
    "axes.labelsize": 24,
    "axes.titlesize": 24,
    "legend.fontsize": 20,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "axes.linewidth": 1.5,
})

# Fig. 2f

In [ ]:
import csv
import os
from tqdm import tqdm
from collections import Counter

# --- 1. DATA PREPARATION: Neuron ID Retrieval ---
def read_neuron_ids(neuron_type, side=None, filename='../data/column_assignment.txt'):
    neuron_ids = []
    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f, delimiter=',')
            for row in reader:
                if row['type'] == neuron_type and (side is None or row['hemisphere'].lower() == side.lower()):
                    try:
                        neuron_ids.append(int(row['root_id']))
                    except ValueError:
                        continue
    except Exception as e:
        print(f"Error reading file {filename}: {str(e)}")
    return neuron_ids


def get_all_neuron_types(filename='../data/column_assignment.txt'):
    neuron_types = set()
    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f, delimiter=',')
            for row in reader:
                neuron_types.add(row['type'])
    except Exception as e:
        print(f"Error reading file {filename}: {str(e)}")
    return list(neuron_types)


neuron_types = get_all_neuron_types()
sides = ['left', 'right']

tasks = {}   # {(type, side): neuron_ids}

for neuron_type in neuron_types:
    for side in sides:
        neuron_ids = read_neuron_ids(neuron_type=neuron_type, side=side)

        tasks[(neuron_type, side)] = neuron_ids

print("\n========== Task summary ==========")
print(f"Total tasks (type × side): {len(tasks)}")

sizes = [len(v) for v in tasks.values()]
print(f"Neuron count per task: min={min(sizes)}, max={max(sizes)}")

empty = sum(1 for v in sizes if v == 0)
print(f"Empty tasks: {empty}")
print("=================================\n")

import matplotlib.pyplot as plt
import scienceplots
import umap
import matplotlib.colors as mcolors
import numpy as np
from sklearn.metrics import silhouette_score

# --- 2. STYLE CONFIGURATION: Publication Standards ---
plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 28,
    "axes.labelsize": 28,
    "axes.titlesize": 28,
    "legend.fontsize": 28,
    "xtick.labelsize": 28,
    "ytick.labelsize": 28,
    "axes.linewidth": 1.5,
})

# --- 3. CORE PROCESSING: Feature Extraction & Standardization ---
def plot_neuron_umap_from_profiles(
    tasks: dict,
    title: str = '',
    crop_size: int = 21,
    n_neighbors: int = 100,
    min_dist: float = 0.7,
    scatter_size: float = 10,
    random_state: int = 42,
    save_name: str = "neuron_umap",
    side_list: list = ['left', 'right', 'both']  
):

    layers_list = ['l1', 'l2', 'l3']
    half = crop_size // 2
    save_dir = '../results/vfp_umap'
    os.makedirs(save_dir, exist_ok=True)

    mask = np.load('./output/combined_eye_mask_41x82.npz')['mask'].astype(bool)
    H, W = mask.shape
    yy, xx = np.where(mask)

    # -------------------------- Load matrices --------------------------
    matrices = {}
    id_maps = {}
    for layer in layers_list:
        matrices[layer] = {}
        id_maps[layer] = {}
        for s in ['left', 'right']:
            path = f'./output/VFM/{layer}_{s}.npz'
            if os.path.exists(path):
                d = np.load(path)
                matrices[layer][s] = {
                    'target_ids': d['target_ids'],
                    'w': (d['exc'] + d['inh']).astype(np.float32)
                }
                id_maps[layer][s] = {nid: i for i, nid in enumerate(d['target_ids'])}
            else:
                matrices[layer][s] = None
                id_maps[layer][s] = {}

    # -------------------------- Collect features --------------------------
    features_dict = {side: {layer: [] for layer in layers_list + ['all']} for side in ['left', 'right', 'both']}
    labels_dict = {side: {layer: [] for layer in layers_list + ['all']} for side in ['left', 'right', 'both']}
    visited = set()

    for (neuron_type, side), neuron_ids in tqdm(tasks.items(), desc='Extracting features'):
        for nid in neuron_ids:
            if nid in visited:
                continue
            visited.add(nid)

            full_layers = []
            max_val = 0.0
            cx = cy = None

            for layer in layers_list:
                full = np.zeros((H, W), dtype=np.float32)
                for s in ['left', 'right']:
                    entry = matrices[layer][s]
                    if entry is None:
                        continue
                    idx = id_maps[layer][s].get(nid)
                    if idx is None:
                        continue
                    w = entry['w'][idx]
                    if s == 'left':
                        full[:, :W//2] = w
                    else:
                        full[:, W//2:] = np.fliplr(w)

                full_layers.append(full)
                flat = full[mask]
                if flat.size == 0:
                    continue
                max_idx = np.argmax(np.abs(flat))
                if np.abs(flat[max_idx]) > max_val:
                    max_val = np.abs(flat[max_idx])
                    cy = yy[max_idx]
                    cx = xx[max_idx]

            if max_val < 1e-12:
                continue

            xs = xx - cx + half
            ys = yy - cy + half
            inside = (xs >= 0) & (xs < crop_size) & (ys >= 0) & (ys < crop_size)

            layer_feats = []
            for full in full_layers:
                patch = np.zeros((crop_size, crop_size), dtype=np.float32)
                patch[ys[inside], xs[inside]] = full[mask][inside]
                layer_feats.append(patch.flatten())

            feat_concat = np.concatenate(layer_feats)

            for i, layer in enumerate(layers_list):
                features_dict[side][layer].append(layer_feats[i])
                labels_dict[side][layer].append(neuron_type)
                features_dict['both'][layer].append(layer_feats[i])
                labels_dict['both'][layer].append(neuron_type)

            features_dict[side]['all'].append(feat_concat)
            labels_dict[side]['all'].append(neuron_type)
            features_dict['both']['all'].append(feat_concat)
            labels_dict['both']['all'].append(neuron_type)

    # --- 4. CLUSTERING & VISUALIZATION ---
    # -------------------------- UMAP + Silhouette Score --------------------------
    embedding_dict = {}
    silhouette_dict = {}
    for side in side_list:
        embedding_dict[side] = {}
        silhouette_dict[side] = {}
        for layer in layers_list + ['all']:
            feats = np.asarray(features_dict[side][layer])
            lbls = np.asarray(labels_dict[side][layer])
            if feats.shape[0] == 0:
                embedding_dict[side][layer] = None
                silhouette_dict[side][layer] = None
                continue

            reducer = umap.UMAP(
                n_neighbors=n_neighbors,
                min_dist=min_dist,
                n_components=2,
                random_state=random_state
            )
            embedding_dict[side][layer] = reducer.fit_transform(feats)

            try:
                score = silhouette_score(
                    feats,
                    lbls,
                    sample_size=min(2000, len(lbls)),
                    random_state=0
                )
            except Exception:
                score = None
            silhouette_dict[side][layer] = score
            if score is not None:
                print(f"Silhouette Score - side: {side}, layer: {layer} -> {score:.3f}")

    # -------------------------- Nature 40-color palette --------------------------
    nature_40 = [
        "#4C72B0","#55A868","#C44E52","#8172B3","#CCB974","#64B5CD",
        "#8C8C8C","#E17C05","#5DA5DA","#B276B2","#F17CB0","#60BD68",
        "#F15854","#4D4D4D","#B2912F","#499894","#E15759","#79706E",
        "#86BCB6","#FF9DA7","#9C755F","#3182bd","#31a354","#74c476","#a1d99b",
        "#756bb1","#9e9ac8","#bcbddc","#636363"
    ]

    # -------------------------- Plot --------------------------
    valid_sides = [s for s in ['left', 'right', 'both'] if s in side_list]
    n_rows = len(valid_sides)
    fig, axes = plt.subplots(n_rows, 4, figsize=(18, 5*n_rows))
    if n_rows == 1:
        axes = np.expand_dims(axes, 0)

    layer_names = ['L1', 'L2', 'L3', 'All']
    scatter_handles = []

    for row_idx, side in enumerate(valid_sides):
        for col_idx, layer_name in enumerate(layer_names):
            ax = axes[row_idx, col_idx]
            key = layer_name.lower() if layer_name != 'All' else 'all'
            emb = embedding_dict[side][key]
            lbls = labels_dict[side][key]
            score = silhouette_dict[side][key]

            if emb is None or len(lbls) == 0:
                ax.axis('off')
                continue

            unique_labels = sorted(set(lbls))
            num_types = len(unique_labels)

            if num_types <= len(nature_40):
                colors_list = nature_40[:num_types]
            else:
                n_colors = len(nature_40)
                colors_list = []
                for i in range(num_types):
                    base_color = nature_40[i % n_colors]
                    rgb = np.array(mcolors.to_rgb(base_color))
                    factor = 0.9 ** (i // 40)
                    colors_list.append(rgb * factor)

            lbls_np = np.array(lbls)
            for i, t in enumerate(unique_labels):
                idx = np.where(lbls_np == t)[0]
                sc = ax.scatter(
                    emb[idx, 0],
                    emb[idx, 1],
                    s=scatter_size,
                    alpha=0.3,
                    color=colors_list[i],
                    edgecolors='none',
                    linewidths=0,
                    label=t if row_idx == n_rows-1 and layer_name == 'All' else None
                )
                if row_idx == n_rows-1 and layer_name == 'All':
                    scatter_handles.append(sc)

            if score is not None:
                ax.set_title(f"{layer_name}\nSilhouette: {score:.3f}")
            else:
                ax.set_title(f"{layer_name}\nSilhouette: N/A")

            ax.set_xticks([])
            ax.set_yticks([])
            ax.axis('off')

    row_labels = {'left':'Left','right':'Right','both':'Both'}
    for i, side in enumerate(valid_sides):
        y = 1 - (i + 0.5)/n_rows
        fig.text(0.02, y, row_labels[side]+title, rotation=90, va='center', ha='center')

    from matplotlib.lines import Line2D
    labels = [s.get_label() for s in scatter_handles]
    colors = [s.get_facecolor()[0] for s in scatter_handles]
    proxy_handles = [Line2D([0], [0], marker='o', color='none', 
                            markerfacecolor=c, alpha=1, markersize=10)
                    for c in colors]
    n_per_col = 60
    N = len(labels)
    import math
    ncol = math.ceil(N / n_per_col)

    new_labels = []
    new_handles = []
    for r in range(n_per_col):
        for c in range(ncol):
            idx = c * n_per_col + r
            if idx < N:
                new_labels.append(labels[idx])
                new_handles.append(proxy_handles[idx])
    
    fig.legend(
        handles=new_handles,
        labels=new_labels,
        loc='center left',
        bbox_to_anchor=(1, 0.5),
        bbox_transform=fig.transFigure,
        frameon=False,
        fontsize=20,
        markerscale=2,
        ncol=ncol
    )

    plt.tight_layout(rect=[0.05, 0.05, 0.93, 0.95])
    plt.savefig(os.path.join(save_dir, f'{save_name}.pdf'), bbox_inches='tight')
    plt.show()

    return embedding_dict, silhouette_dict

# --- 5. EXECUTION ---
# -------------------------------
# UMAP
# -------------------------------
plot_neuron_umap_from_profiles(
    tasks=tasks,
    crop_size=5,
    save_name="column_umap"
)

# Supplementary Fig. 4

In [ ]:
import csv
import os
from tqdm import tqdm
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import scienceplots


# --------------------------
# Style Configuration
# --------------------------
plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"], 
    "font.size": 18,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "legend.fontsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "axes.linewidth": 1.5,
})

# --------------------------
# Data Preparation
# --------------------------
def read_neuron_ids(neuron_type, side=None, filename='../data/column_assignment.txt'):
    neuron_ids = []
    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f, delimiter=',')
            for row in reader:
                if row['type'] == neuron_type and (side is None or row['hemisphere'].lower() == side.lower()):
                    try:
                        neuron_ids.append(int(row['root_id']))
                    except ValueError:
                        continue
    except Exception as e:
        print(f"Error reading file {filename}: {str(e)}")
    return neuron_ids


def get_all_neuron_types(filename='../data/column_assignment.txt'):
    neuron_types = set()
    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f, delimiter=',')
            for row in reader:
                neuron_types.add(row['type'])
    except Exception as e:
        print(f"Error reading file {filename}: {str(e)}")
    return list(neuron_types)

neuron_types = get_all_neuron_types()
sides = ['left', 'right']

tasks = {}   # {(type, side): neuron_ids}

for neuron_type in neuron_types:
    for side in sides:
        neuron_ids = read_neuron_ids(neuron_type=neuron_type, side=side)

        tasks[(neuron_type, side)] = neuron_ids

print("\n========== Task summary ==========")
print(f"Total tasks (type × side): {len(tasks)}")

sizes = [len(v) for v in tasks.values()]
print(f"Neuron count per task: min={min(sizes)}, max={max(sizes)}")

empty = sum(1 for v in sizes if v == 0)
print(f"Empty tasks: {empty}")
print("=================================\n")

# ------------------------------------
# Feature Extraction & Standardization
# ------------------------------------
def plot_type_mean_correlation(
    tasks: dict,
    crop_size: int = 21,
    save_name: str = "type_mean_correlation_advanced"
):

    layers_list = ['l1', 'l2', 'l3']
    layer_names = ['L1', 'L2', 'L3', 'All']
    half = crop_size // 2

    save_dir = '../results/vfp_correlation'
    os.makedirs(save_dir, exist_ok=True)

    mask = np.load('./output/combined_eye_mask_41x82.npz')['mask'].astype(bool)
    H, W = mask.shape
    yy, xx = np.where(mask)

    # --------------------------
    # Load matrices
    # --------------------------
    matrices = {}
    for layer in layers_list:
        matrices[layer] = {}
        for s in ['left', 'right']:
            path = f'./output/VFM/{layer}_{s}.npz'
            if os.path.exists(path):
                d = np.load(path)
                matrices[layer][s] = {
                    'target_ids': d['target_ids'],
                    'w': (d['exc'] + d['inh']).astype(np.float32)
                }
            else:
                matrices[layer][s] = None

    # --------------------------
    # Feature extraction
    # --------------------------
    features_dict = {side: {layer: {} for layer in layers_list + ['all']}
                    for side in ['left', 'right', 'both']}

    visited = set()
    for (neuron_type, side), neuron_ids in tqdm(tasks.items(), desc="Extracting"):
        for nid in neuron_ids:
            if nid in visited:
                continue
            visited.add(nid)

            full_layers = []
            max_val = 0.0
            cx = cy = None

            for layer in layers_list:
                full = np.zeros((H, W), dtype=np.float32)
                for s in ['left', 'right']:
                    entry = matrices[layer][s]
                    if entry is None:
                        continue
                    ids = entry['target_ids']
                    idx = np.where(ids == nid)[0]
                    if idx.size == 0:
                        continue
                    w = entry['w'][idx[0]]
                    if s == 'left':
                        full[:, :W//2] = w
                    else:
                        full[:, W//2:] = np.fliplr(w)
                full_layers.append(full)

                abs_valid = np.abs(full)[mask]
                if abs_valid.size == 0:
                    continue
                if abs_valid.max() > max_val:
                    max_val = abs_valid.max()
                    idx_max = np.argmax(abs_valid)
                    cy = yy[idx_max]
                    cx = xx[idx_max]

            if max_val < 1e-12:
                continue

            xs = xx - cx + half
            ys = yy - cy + half
            inside = (xs >= 0) & (xs < crop_size) & (ys >= 0) & (ys < crop_size)

            layer_feats = []
            for full in full_layers:
                patch = np.zeros((crop_size, crop_size), dtype=np.float32)
                patch[ys[inside], xs[inside]] = full[mask][inside]
                layer_feats.append(patch.flatten())

            feat_concat = np.concatenate(layer_feats)

            for i, layer in enumerate(layers_list):
                for s in [side, 'both']:
                    features_dict[s][layer].setdefault(neuron_type, []).append(layer_feats[i])
            for s in [side, 'both']:
                features_dict[s]['all'].setdefault(neuron_type, []).append(feat_concat)


        
    # --------------------------
    # Compute correlations
    # --------------------------
    corr_dict = {side: {layer: {} for layer in layers_list + ['all']}
                 for side in ['left', 'right', 'both']}

    for side in ['left', 'right', 'both']:
        for layer in layers_list + ['all']:
            for t, feats in features_dict[side][layer].items():
                feats = np.array(feats)
                if np.all(feats == 0):
                    corr_dict[side][layer][t] = [0] * len(feats)
                    continue
                mean_pattern = np.mean(feats, axis=0)
                corrs = []
                for v in feats:
                    if np.std(v) == 0 or np.std(mean_pattern) == 0:
                        corrs.append(0) 
                    else:
                        corrs.append(np.corrcoef(v, mean_pattern)[0, 1])
                corr_dict[side][layer][t] = corrs

    # --------------------------
    # Plot
    # --------------------------
    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
    palette = ['#4C72B0', '#55A868', '#C44E52', '#8172B3']
    width = 0.18

    for row_idx, side in enumerate(['left', 'right', 'both']):
        ax = axes[row_idx]
        types = sorted(corr_dict[side]['all'].keys())
        x_base = np.arange(len(types))

        for i, layer in enumerate(['l1', 'l2', 'l3', 'all']):
            offsets = x_base + (i - 1.5) * width
            data = [corr_dict[side][layer].get(t, [0]) for t in types]

            # violin
            parts = ax.violinplot(
                data,
                positions=offsets,
                widths=width,
                showmeans=False,
                showmedians=False,
                showextrema=False
            )
            for pc in parts['bodies']:
                pc.set_facecolor(palette[i])
                pc.set_alpha(0.3)

            # box
            box = ax.boxplot(
                data,
                positions=offsets,
                widths=width * 0.6,
                patch_artist=True,
                showfliers=False
            )
            for patch in box['boxes']:
                patch.set_facecolor("white")
                patch.set_edgecolor(palette[i])
                patch.set_linewidth(1.2)

            # jitter scatter
            for j, vals in enumerate(data):
                if len(vals) == 0:
                    continue
                x_jitter = np.random.normal(offsets[j], width * 0.08, size=len(vals))
                ax.scatter(x_jitter, vals,
                           s=10,
                           alpha=0.05,
                           color=palette[i],
                           edgecolors='none')

        ax.set_ylim(-1, 1)
        ax.set_ylabel("Pearson r")
        ax.set_title(side.capitalize())

    for i, layer in enumerate(['L1', 'L2', 'L3', 'All']):
        axes[0].scatter([], [], color=palette[i], label=layer, alpha=0.6, s=50)

    axes[0].legend(
        title="Layer",
        fontsize=16,
        title_fontsize=18,
        loc='center left',
        bbox_to_anchor=(1.02, 0.5), 
        borderaxespad=0,
        markerscale=3
    )

    axes[-1].set_xticks(np.arange(len(types)))
    axes[-1].set_xticklabels(types, rotation=45, ha='right')

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{save_name}.pdf"), bbox_inches='tight')
    plt.show()

plot_type_mean_correlation(
    tasks=tasks,
    crop_size=5,
    save_name="heatmap"
)

# 

# Supplementary Fig. 3c,d,e,f

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import csv
from scipy import stats
from matplotlib.gridspec import GridSpec
from matplotlib.path import Path
from matplotlib.cm import ScalarMappable

# -----------------------------
# CONFIG
# -----------------------------
RESULTS_DIR = './output/DSI/side_separated/'
FRI_CSV_FILE = './output/FRI/FRI_all_neurons_per_neuron_baseline.csv'

SIDES = ['left', 'right']
CSV_FILES = {
    'left': {
        'D2L': os.path.join(RESULTS_DIR, 'DSI_OFF_left.csv'),
        'L2D': os.path.join(RESULTS_DIR, 'DSI_ON_left.csv'),
    },
    'right': {
        'D2L': os.path.join(RESULTS_DIR, 'DSI_OFF_right.csv'),
        'L2D': os.path.join(RESULTS_DIR, 'DSI_ON_right.csv'),
    }
}

# Visualization parameters
SPACING = 1.2
MAX_DENSITY_WIDTH = 0.12
Y_GRID_POINTS = 400
SCATTER_JITTER = 0.02
PAGE_SIZE = 25  

# -----------------------------
# DATA UTILITIES
# -----------------------------

# Ensures output directories exist
def ensure_dirs():
    os.makedirs(RESULTS_DIR, exist_ok=True)

# Computes median and 95% confidence intervals (2.5th to 97.5th percentile)
def compute_stats(values):
    values = np.array(values)
    median = np.median(values)
    ci_low, ci_high = np.percentile(values, [2.5, 97.5])
    return median, ci_low, ci_high


# Reads FRI values grouped by neuron type
def read_fri_csv(csv_file=FRI_CSV_FILE):
    types_fri = {}
    with open(csv_file, 'r', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            t = row['type']
            v = float(row['fri'])
            types_fri.setdefault(t, []).append(v)
    return types_fri

# Reads DSI and Tuning Angle data with multi-encoding support
def read_csv_with_angle(csv_path):
    data_dsi = {}
    data_angle = {}
    encodings = ['utf-8', 'utf-8-sig', 'gbk', 'ansi', 'latin1']
    for enc in encodings:
        try:
            with open(csv_path, 'r', encoding=enc, newline='') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    t = row['type']
                    try:
                        v = float(row['DSI'])
                        a = float(row.get('PreferredAngle', -1))
                    except Exception:
                        continue
                    v = max(0.0, min(1.0, v))
                    data_dsi.setdefault(t, []).append(v)
                    data_angle.setdefault(t, []).append(a)
            return data_dsi, data_angle
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f"{csv_path}")

# ------------------------------------
# CORE PROCESSING: Statistical Density
# ------------------------------------

# Computes normalized Gaussian KDE for violin-style distribution plots
def safe_kde_density(vals, y_grid):
    vals = np.asarray(vals)
    if len(vals) == 0:
        return np.zeros_like(y_grid)
    if np.allclose(vals.std(), 0.0):
        center = float(vals[0])
        sigma = 0.01
        density = np.exp(-0.5 * ((y_grid - center) / sigma) ** 2)
        if np.max(density) == 0:
            return np.zeros_like(y_grid)
        density /= np.max(density)
        return density
    kde = stats.gaussian_kde(vals)
    density = kde(y_grid)
    if np.max(density) == 0:
        return np.zeros_like(y_grid)
    density /= np.max(density)
    return density

# Bins angular data and sums DSI for polar distribution analysis
def angle_bin_count_and_sum_dsi(ang_vals_deg, dsi_vals, step=20):
    if len(ang_vals_deg) == 0:
        bins = np.deg2rad(np.arange(0, 360 + step, step))
        bin_centers = (bins[:-1] + bins[1:]) / 2
        return np.zeros_like(bin_centers), np.zeros_like(bin_centers), bin_centers

    ang_vals = np.deg2rad(np.array(ang_vals_deg))
    dsi_vals = np.array(dsi_vals)
    
    bins = np.deg2rad(np.arange(0, 360 + step, step))
    bin_centers = (bins[:-1] + bins[1:]) / 2
    
    counts = np.zeros_like(bin_centers)
    sum_dsi = np.zeros_like(bin_centers)
    
    for j in range(len(bins) - 1):
        mask = (ang_vals >= bins[j]) & (ang_vals < bins[j + 1])
        counts[j] = np.sum(mask)
        sum_dsi[j] = np.sum(dsi_vals[mask])
    
    counts = np.append(counts, counts[0])
    sum_dsi = np.append(sum_dsi, sum_dsi[0])
    bin_centers = np.append(bin_centers, bin_centers[0])
    
    return counts, sum_dsi, bin_centers

# -------------------------------
# VISUALIZATION: Integrated Atlas
# -------------------------------

def sharp_triangle_marker(rotation_deg=0):
    verts = np.array([
        [0.0, 0.0],
        [-0.2, -0.5],
        [0.2, -0.5],
        [0.0, 0.0]
    ])
    theta = np.deg2rad(rotation_deg)
    rot = np.array([[np.cos(theta), -np.sin(theta)],
                    [np.sin(theta), np.cos(theta)]])
    verts = verts @ rot.T
    return Path(verts)

deep_red = '#8B0000'   # Nature vermillion
deep_blue = '#1E3A8A'  # Nature cobalt blue
ci_color = '#808080'  # Gray for confidence intervals
median_color = '#000000'  # Black for median lines
bg_color = '#FFFFFF'  # White background
ax_facecolor = '#FFFFFF'  # White for axes

def plot_combined_fri_dsi():
    ensure_dirs()
    types_fri = read_fri_csv(FRI_CSV_FILE)

    for side in SIDES:
        d2l_dsi, d2l_angle = read_csv_with_angle(CSV_FILES[side]['D2L'])
        l2d_dsi, l2d_angle = read_csv_with_angle(CSV_FILES[side]['L2D'])

        all_types = sorted(set(types_fri.keys()) | set(d2l_dsi.keys()) | set(l2d_dsi.keys()))
        priority = ['C3', 'L5', 'Mi1', 'Mi4', 'T4a', 'T4b', 'T4c', 'T4d', 'Tm3',
                    'L1','L2','L3','L4','Mi9','T5a','T5b','T5c','T5d','Tm1','Tm2','Tm4','Tm9',
                    'Dm3p','Dm3q','Dm3v']
        sorted_types = [t for t in priority if t in all_types] + [t for t in all_types if t not in priority]
        filtered_types = [t for t in sorted_types if
                          len(types_fri.get(t, [])) >= 10 or
                          len(d2l_dsi.get(t, [])) >= 10 or
                          len(l2d_dsi.get(t, [])) >= 10]

        n_types = len(filtered_types)
        n_pages = (n_types + PAGE_SIZE - 1) // PAGE_SIZE

        for page in range(n_pages):
            start = page * PAGE_SIZE
            end = min((page+1) * PAGE_SIZE, n_types)
            sub_types = filtered_types[start:end]
            x = np.arange(len(sub_types), dtype=float) * SPACING

            fig_w = max(12, len(sub_types) * SPACING)
            fig_h = 16
            fig = plt.figure(figsize=(fig_w, fig_h), facecolor='white')
            gs = GridSpec(4, 1, figure=fig, height_ratios=[1,1,1,1], hspace=0.25)

            # 1. FRI Subplot
            ax_fri = fig.add_subplot(gs[0,0])
            ax_fri.set_facecolor('white')
            y_grid_fri = np.linspace(-1,1,Y_GRID_POINTS)

            group_green = {'L5', 'C3', 'Mi1', 'Mi4', 'T4a', 'T4b', 'T4c', 'T4d', 'Tm3'}
            group_red = {'L1','L2','L3','L4','Mi9','T5a','T5b','T5c','T5d','Tm1','Tm2','Tm4','Tm9'}
            group_priority = {'Dm3p','Dm3q','Dm3v'}
            green_color = '#228B22'
            red_color = '#B22222'
            gray_color = '#A9A9A9'
            priority_color = 'purple'
            ci_color = '#6A5ACD'
            median_color = 'black'

            for i, t in enumerate(sub_types):
                vals = np.array(types_fri.get(t, []))
                if len(vals) < 10: continue
                density = safe_kde_density(vals, y_grid_fri)
                width = 0.18*density
                if t in group_green: fill_color = green_color
                elif t in group_red: fill_color = red_color
                elif t in group_priority: fill_color = priority_color
                else: fill_color = gray_color
                ax_fri.fill_betweenx(y_grid_fri, x[i]-width, x[i]+width, facecolor=fill_color, alpha=0.5, zorder=2)
                median, low, high = compute_stats(vals)
                ax_fri.plot([x[i], x[i]], [low,high], color=ci_color,lw=1.5,zorder=3)
                cap_w = 0.08
                ax_fri.plot([x[i]-cap_w, x[i]+cap_w],[low,low],color=ci_color,lw=1.5,zorder=3)
                ax_fri.plot([x[i]-cap_w, x[i]+cap_w],[high,high],color=ci_color,lw=1.5,zorder=3)
                ax_fri.plot([x[i]-0.12, x[i]+0.12],[median,median],color=median_color,lw=2.2,zorder=4)

            ax_fri.axhline(0,color='gray',ls='--',lw=0.8)
            ax_fri.tick_params(labelbottom=False)
            ax_fri.set_ylabel('FRI (ON-OFF)')
            ax_fri.set_ylim(-1,1)

            ax_fri = fig.add_subplot(gs[0,0])
            ax_fri.set_facecolor('white')
            y_grid_fri = np.linspace(-1,1,Y_GRID_POINTS)

            group_red = {'L5', 'C3', 'Mi1', 'Mi4', 'T4a', 'T4b', 'T4c', 'T4d', 'Tm3'}
            group_blue = {'L1','L2','L3','L4','Mi9','T5a','T5b','T5c','T5d','Tm1','Tm2','Tm4','Tm9'} 
            group_priority = {'Dm3p','Dm3q','Dm3v'}

            blue_subtypes = [t for t in sub_types if t in group_blue]
            last_blue = blue_subtypes[-1] if blue_subtypes else None

            for i, t in enumerate(sub_types):
                left = (i+0.3) / (len(sub_types))
                right = ((i+1.3)) / (len(sub_types))

                if t in group_red:
                    ax_fri.axhspan(0, 1, facecolor='#FFCCCC', alpha=0.3, xmin=left, xmax=right, zorder=0)
                elif t in group_blue and t != last_blue:
                    ax_fri.axhspan(-1, 0, facecolor='#A7C7E7', alpha=0.3, xmin=left, xmax=right, zorder=0)

            for i, t in enumerate(sub_types):
                vals = np.array(types_fri.get(t, []))
                if len(vals) < 10: 
                    continue
                density = safe_kde_density(vals, y_grid_fri)
                width = 0.18 * density

                if t in group_red:
                    fill_color = deep_red
                elif t in group_blue:
                    fill_color = deep_blue
                elif t in group_priority:
                    fill_color = priority_color
                else:
                    fill_color = gray_color

                ax_fri.fill_betweenx(y_grid_fri, x[i]-width, x[i]+width, facecolor=fill_color, alpha=0.5, zorder=2)

                median, low, high = compute_stats(vals)
                ax_fri.plot([x[i], x[i]], [low, high], color=ci_color, lw=1.5, zorder=3)
                cap_w = 0.08
                ax_fri.plot([x[i]-cap_w, x[i]+cap_w], [low, low], color=ci_color, lw=1.5, zorder=3)
                ax_fri.plot([x[i]-cap_w, x[i]+cap_w], [high, high], color=ci_color, lw=1.5, zorder=3)
                ax_fri.plot([x[i]-0.12, x[i]+0.12], [median, median], color=median_color, lw=2.2, zorder=4)

            ax_fri.axhline(0, color='gray', ls='--', lw=0.8)
            ax_fri.tick_params(labelbottom=False)
            ax_fri.set_ylabel('FRI (ON-OFF)')
            ax_fri.set_ylim(-1, 1)

            if any(t in group_red for t in sub_types):
                red_indices = [i for i, t in enumerate(sub_types) if t in group_red]
                last_red_idx = red_indices[-1]
                ax_fri.text(x[last_red_idx], 0.85, 'Known ON-contrast selective',
                            color=deep_red, ha='right', va='center', fontweight='bold')

            if any(t in group_blue for t in sub_types):
                blue_indices = [i for i, t in enumerate(sub_types) if t in group_blue]
                last_blue_idx = blue_indices[-1]
                ax_fri.text(x[last_blue_idx], -0.85, 'Known OFF-contrast selective',
                            color=deep_blue, ha='right', va='center', fontweight='bold')

            ax = fig.add_subplot(gs[1,0], sharex=ax_fri)
            ax.set_facecolor(ax_facecolor)
            ax.set_ylim(-1.0, 1.0)
            ax.set_yticks(np.arange(-1.0, 1.01, 0.25))
            ax.set_yticklabels([f'{np.abs(v):.2f}' if v != 0 else '0' for v in np.arange(-1.0, 1.01, 0.25)])

            ax.axhline(0, color='gray', ls='--', lw=0.8)
            ax.set_ylabel('DSI(Right)')
            for spine in ['left', 'bottom', 'right', 'top']:
                ax.spines[spine].set_linewidth(1.5)
            ax.tick_params(labelbottom=False)
            max_ref_y = 1 * 0.9  
            ax.hlines(y=max_ref_y, xmin=x[-1]-0.85, xmax=x[-1]-0.55, color="black", lw=2, zorder=6)
            ax.text(x[-1]-0.5, max_ref_y, 'Median', color="black", ha='left', va='center', zorder=6)

            y_grid = np.linspace(0.0, 1.0, Y_GRID_POINTS)
            
            for i, t in enumerate(sub_types):
                # Light→Dark (red)
                vals_red = l2d_dsi.get(t, [])
                n_red = len(vals_red)
                angles_l2d = [a for a in l2d_angle.get(t, []) if a >= 0]
                dsis_l2d = [v for (v, a) in zip(l2d_dsi.get(t, []), l2d_angle.get(t, [])) if a >= 0]

                angles_d2l = [a for a in d2l_angle.get(t, []) if a >= 0]
                dsis_d2l = [v for (v, a) in zip(d2l_dsi.get(t, []), d2l_angle.get(t, [])) if a >= 0]
                if n_red >= 10:
                    density = safe_kde_density(vals_red, y_grid)
                    width = MAX_DENSITY_WIDTH * density
                    ax.fill_betweenx(y_grid, x[i]-width, x[i]+width, facecolor=deep_red, alpha=0.5, edgecolor='none', zorder=2)
                    median, low, high = compute_stats(vals_red)
                    ci_w = 0.08
                    median_w = ci_w * 1.2
                    ax.plot([x[i], x[i]], [low, high], color=ci_color, lw=1.5, zorder=3)
                    ax.plot([x[i]-ci_w, x[i]+ci_w], [low, low], color=ci_color, lw=1.5, zorder=3)
                    ax.plot([x[i]-ci_w, x[i]+ci_w], [high, high], color=ci_color, lw=1.5, zorder=3)
                    ax.plot([x[i]-median_w, x[i]+median_w], [median, median], color="black", lw=2, zorder=4)
                    jitter = np.random.uniform(-SCATTER_JITTER, SCATTER_JITTER, n_red)

                # Dark→Light (blue)
                vals_blue = d2l_dsi.get(t, [])
                n_blue = len(vals_blue)
                if n_blue >= 10:
                    density = safe_kde_density(vals_blue, y_grid)
                    width = MAX_DENSITY_WIDTH * density
                    ax.fill_betweenx(-y_grid, x[i]-width, x[i]+width, facecolor=deep_blue, alpha=0.5, edgecolor='none', zorder=2)
                    median, low, high = compute_stats(vals_blue)
                    ci_w = 0.08
                    median_w = ci_w * 1.2
                    ax.plot([x[i], x[i]], [-low, -high], color=ci_color, lw=1.5, zorder=3)
                    ax.plot([x[i]-ci_w, x[i]+ci_w], [-low, -low], color=ci_color, lw=1.5, zorder=3)
                    ax.plot([x[i]-ci_w, x[i]+ci_w], [-high, -high], color=ci_color, lw=1.5, zorder=3)
                    ax.plot([x[i]-median_w, x[i]+median_w], [-median, -median], color="black", lw=2, zorder=4)
                    jitter = np.random.uniform(-SCATTER_JITTER, SCATTER_JITTER, n_blue)
                        
            # ----------------------------
            # Middle Part: PreferredAngle KDE Dual-axis
            # ----------------------------
            ax2 = fig.add_subplot(gs[2,0], sharex=ax_fri)

            ax2.set_facecolor(ax_facecolor)
            ax2.set_ylim(-360, 360)
            ax2.set_yticks(np.arange(-360, 361, 90))
            ax2.set_yticklabels([f'{np.abs(v):.0f}' for v in np.arange(-360, 361, 90)])
            ax2.axhline(0, color='gray', ls='--', lw=0.8)
            ax2.set_ylabel('PreferredAngle(°)')
            for spine in ['left', 'bottom', 'right', 'top']:
                ax2.spines[spine].set_linewidth(1.5)
            ax2.tick_params(axis='both', which='major', direction='out', length=6, width=1.5)
            ax2.set_xlabel('')
            max_ref_y = 360 * 0.9  
            ax2.hlines(y=max_ref_y, xmin=x[-1]-0.85, xmax=x[-1]-0.55, color='black', lw=2, zorder=6)
            ax2.text(x[-1]-0.5, max_ref_y, 'Max Density', color='black', ha='left', va='center', zorder=6)


            y_grid_angle = np.linspace(0, 360, Y_GRID_POINTS)
            for i, t in enumerate(sub_types):
                vals_angle_l2d = np.array([v for v in l2d_angle.get(t, []) if v >= 0])
                vals_angle_d2l = np.array([v for v in d2l_angle.get(t, []) if v >= 0])
                n_vals = max(len(l2d_dsi.get(t, [])), 1)
                # Light→Dark
                if n_red >= 10:
                    norm_red_width = np.maximum(len(vals_angle_l2d) / n_vals, 0.01)
                    density = safe_kde_density(vals_angle_l2d, y_grid_angle)
                    width = MAX_DENSITY_WIDTH * density * norm_red_width
                    ax2.fill_betweenx(y_grid_angle, x[i]-width, x[i]+width, facecolor=deep_red, alpha=0.5, edgecolor='none', zorder=2)
                    jitter = np.random.uniform(-SCATTER_JITTER, SCATTER_JITTER, len(vals_angle_l2d))
                    max_density = np.max(density)
                    max_density_line = np.where(density == max_density)[0][0]
                    ax2.plot([x[i]-MAX_DENSITY_WIDTH * norm_red_width, x[i]+MAX_DENSITY_WIDTH * norm_red_width], [y_grid_angle[max_density_line], y_grid_angle[max_density_line]], color='black', lw=2, zorder=3)
                        
                # Dark→Light
                if n_blue >= 10:
                    norm_blue_width = np.maximum(len(vals_angle_d2l) / n_vals, 0.01)
                    density = safe_kde_density(vals_angle_d2l, y_grid_angle)
                    width = MAX_DENSITY_WIDTH * density * norm_blue_width
                    ax2.fill_betweenx(-y_grid_angle, x[i]-width, x[i]+width, facecolor=deep_blue, alpha=0.5, edgecolor='none', zorder=2)
                    jitter = np.random.uniform(-SCATTER_JITTER, SCATTER_JITTER, len(vals_angle_d2l))
                    max_density = np.max(density)
                    max_density_line = np.where(density == max_density)[0][0]
                    ax2.plot([x[i]-MAX_DENSITY_WIDTH * norm_blue_width, x[i]+MAX_DENSITY_WIDTH * norm_blue_width], [-y_grid_angle[max_density_line], -y_grid_angle[max_density_line]], color='black', lw=2, zorder=3)
                        
                        
                ax2.set_xticks(x)
                ax2.set_xticklabels(['']*len(sub_types))  

                for i, t in enumerate(sub_types):
                    color = deep_red if t in group_red else deep_blue if t in group_blue else 'black'
                    fontweight = 'bold' if t.startswith('T4') or t.startswith('T5') else 'normal'
                    ax2.text(x[i], -370, t, rotation=90, ha='center', va='top', color=color, fontweight=fontweight)

            # ----------------------------
            # Bottom Part: Radar per type
            # ----------------------------
            n_radar_rows = 2  
            gs_radar = GridSpec(
                n_radar_rows,
                len(sub_types)+1, 
                figure=fig,
                top=0.29,
                bottom=0.05,
                left=0.15,
                right=0.903,
                hspace=0,
                wspace=0
            )

            angle_step = 20
            dsi_step = 0.1
            angle_bins = np.deg2rad(np.arange(0, 360 + angle_step, angle_step))
            dsi_bins = np.arange(0, 1 + dsi_step, dsi_step)

            hist_data = {}
            angle_bin_curves_red = {}
            angle_bin_curves_blue = {}

            for t in sub_types:
                vals_angle_l2d = np.array([v for v in l2d_angle.get(t, []) if v >= 0])
                vals_angle_d2l = np.array([v for v in d2l_angle.get(t, []) if v >= 0])
                dsis_l2d = [v for (v, a) in zip(l2d_dsi.get(t, []), l2d_angle.get(t, [])) if a >= 0]
                dsis_d2l = [v for (v, a) in zip(d2l_dsi.get(t, []), d2l_angle.get(t, [])) if a >= 0]

                counts_r, sum_dsi_r, bin_centers_r = angle_bin_count_and_sum_dsi(vals_angle_l2d, dsis_l2d, step=angle_step)
                angle_bin_curves_red[t] = (sum_dsi_r, bin_centers_r)

                # 🔵 Blue
                counts_b, sum_dsi_b, bin_centers_b = angle_bin_count_and_sum_dsi(vals_angle_d2l, dsis_d2l, step=angle_step)
                angle_bin_curves_blue[t] = (sum_dsi_b, bin_centers_b)

                # Red heatmap
                vals_red = np.array(l2d_dsi.get(t, []))
                angles_red = np.deg2rad(np.array(l2d_angle.get(t, [])))
                mask_red = vals_red > 0
                H_red, _, _ = np.histogram2d(angles_red[mask_red], vals_red[mask_red],
                                            bins=[angle_bins, dsi_bins])
                H_red = H_red.T
                hist_data[(t, 'red')] = H_red

                # Blue heatmap
                vals_blue = np.array(d2l_dsi.get(t, []))
                angles_blue = np.deg2rad(np.array(d2l_angle.get(t, [])))
                mask_blue = vals_blue > 0
                H_blue, _, _ = np.histogram2d(angles_blue[mask_blue], vals_blue[mask_blue],
                                            bins=[angle_bins, dsi_bins])
                H_blue = H_blue.T
                hist_data[(t, 'blue')] = H_blue

            max_red_heatmap = max(np.max(H) for (t, color), H in hist_data.items() if color=='red')
            max_blue_heatmap = max(np.max(H) for (t, color), H in hist_data.items() if color=='blue')

            max_red_curve = max(np.max(c[0]) for c in angle_bin_curves_red.values())
            max_blue_curve = max(np.max(c[0]) for c in angle_bin_curves_blue.values())

            for i, t in enumerate(sub_types):
                ax_r = fig.add_subplot(gs_radar[0, i], polar=True)
                ax_b = fig.add_subplot(gs_radar[1, i], polar=True)

                H_red = hist_data[(t, 'red')] / max_red_heatmap  
                for ai in range(H_red.shape[1]):
                    for di in range(H_red.shape[0]):
                        if H_red[di, ai] == 0:
                            continue
                        theta = (angle_bins[ai] + angle_bins[ai+1])/2
                        width = angle_bins[ai+1] - angle_bins[ai]
                        r = (dsi_bins[di] + dsi_bins[di+1])/2
                        height = dsi_bins[di+1] - dsi_bins[di]
                        color = plt.cm.Reds(H_red[di, ai])
                        ax_r.bar(theta, height, width=width, bottom=r - height/2, color=color, edgecolor=None, alpha=0.8)

                curve_r, bin_centers_r = angle_bin_curves_red[t]
                ax_r.plot(bin_centers_r, curve_r / max_red_curve, color='darkred', lw=1.2, zorder=10, alpha=0.9)
                ax_r.set_xticks([]); ax_r.set_yticks([]); ax_r.set_ylim(0,1)

                H_blue = hist_data[(t, 'blue')] / max_blue_heatmap
                for ai in range(H_blue.shape[1]):
                    for di in range(H_blue.shape[0]):
                        if H_blue[di, ai] == 0:
                            continue
                        theta = (angle_bins[ai] + angle_bins[ai+1])/2
                        width = angle_bins[ai+1] - angle_bins[ai]
                        r = (dsi_bins[di] + dsi_bins[di+1])/2
                        height = dsi_bins[di+1] - dsi_bins[di]
                        color = plt.cm.Blues(H_blue[di, ai])
                        ax_b.bar(theta, height, width=width, bottom=r - height/2, color=color, edgecolor=None, alpha=0.8)

                curve_b, bin_centers_b = angle_bin_curves_blue[t]
                ax_b.plot(bin_centers_b, curve_b / max_blue_curve, color='darkblue', lw=1.2, zorder=10, alpha=0.9)
                ax_b.set_xticks([]); ax_b.set_yticks([]); ax_b.set_ylim(0,1)

            
            # -------------------------------------------
            # REFERENCE COORDINATES: Orientation Guide
            # -------------------------------------------
            ref_ax = fig.add_axes([0.1, 0.12, 0.04, 0.08], polar=True)
            ref_ax.set_theta_zero_location('N')
            ref_ax.set_theta_direction(-1)
            ref_ax.set_ylim(0, 1)
            ref_ax.set_yticks([])
            ref_ax.set_xticks(np.deg2rad([0, 90, 180, 270]))
            ref_ax.set_xticklabels(['90', '0', '270', '180'])
            ref_ax.tick_params(pad=5)
            ref_ax.grid(True, linestyle='--', linewidth=1, alpha=0.7)
            for spine in ref_ax.spines.values():
                spine.set_visible(False)
            ref_ax.text(0, 1.7, 'Direction (°)', ha='center', va='center', fontweight='bold')
            ref_ax.set_facecolor("lightgray")

            from matplotlib.cm import ScalarMappable
            from mpl_toolkits.axes_grid1 import make_axes_locatable
            
            cax_r = fig.add_axes([0.89, 0.20, 0.01, 0.06])  # [left, bottom, width, height]
            sm_red = ScalarMappable(cmap='Reds', norm=plt.Normalize(0, 1))
            cbar_red = fig.colorbar(sm_red, cax=cax_r, orientation='vertical')
            cbar_red.set_label('Relative counts')
            cax_r.set_yticks([0, 1])
            cax_r.set_xticks([])

            cax_b = fig.add_axes([0.89, 0.08, 0.01, 0.06]) 
            sm_blue = ScalarMappable(cmap='Blues', norm=plt.Normalize(0, 1))
            cbar_blue = fig.colorbar(sm_blue, cax=cax_b, orientation='vertical')
            cax_b.set_yticks([0, 1])
            cax_b.set_xticks([])

            out_path = os.path.join(RESULTS_DIR, f'{side}_combined_page_{page+1}.png')
            plt.savefig(out_path, dpi=200)
            plt.close(fig)
            print(f"✅ Saved page {page+1} -> {out_path}")

if __name__ == '__main__':
    plot_combined_fri_dsi()

# Fig. 4

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import csv
import scienceplots

# -----------------------------
# CONFIG & STYLE SETUP
# -----------------------------
plt.style.use(['science', 'nature', 'no-latex'])
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],  
    "font.size": 20,
    "axes.labelsize": 24,
    "axes.titlesize": 50,
    "legend.fontsize": 20,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "axes.linewidth": 1.5,
})

# Path and category definitions
RESULTS_DIR = '../results/DSI/circle_max_angle/'
os.makedirs(RESULTS_DIR, exist_ok=True)

CSV_FILES_OFF = {
    'left': './output/DSI/DSI_OFF_left.csv',
    'right': './output/DSI/DSI_OFF_right.csv'
}
CSV_FILES_ON = {
    'left': './output/DSI/DSI_ON_left.csv',
    'right': './output/DSI/DSI_ON_right.csv'
}

T4_TYPES = ['T4a','T4b','T4c','T4d']
T5_TYPES = ['T5a','T5b','T5c','T5d']
OFF_EXTRA_TYPES = ['HSN','HSE','HSS','VS1','VS2','VS3','VS4','VS5','VS6','VS7','VS8']

# Ground truth experimental tuning angles for validation.
EXPERIMENT_ANGLES = {
    'T4a': {'left':180,'right':0},
    'T4b': {'left':0,'right':180},
    'T4c': {'left':90,'right':90},
    'T4d': {'left':270,'right':270},
    'T5a': {'left':180,'right':0},
    'T5b': {'left':0,'right':180},
    'T5c': {'left':90,'right':90},
    'T5d': {'left':270,'right':270},
    'VS1': {'left':270,'right':270},
    'VS2': {'left':270,'right':270},
    'VS3': {'left':270,'right':270},
    'VS4': {'left':270,'right':270},
    'VS5': {'left':270,'right':270},
    'VS6': {'left':270,'right':270},
    'VS7': {'left':270,'right':270},
    'VS8': {'left':270,'right':270},
    'HSN': {'left':180,'right':0},
    'HSE': {'left':180,'right':0},
    'HSS': {'left':180,'right':0},
}

# -----------------------------
# DATA PROCESSING UTILITIES
# -----------------------------
def read_csv_target_types(csv_path, target_types):
    data = {t: [] for t in target_types}
    with open(csv_path, 'r', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            t = row['type']
            if t not in target_types:
                continue
            try:
                angle = float(row.get('PreferredAngle', -1))
                if angle < 0:
                    continue
                dsi = float(row['DSI'])
                data[t].append((angle, dsi))
            except:
                continue
    return data

def get_max_dsi_group_angle(neurons, group_size=10):
    if len(neurons) == 0:
        return None
    bins = np.arange(0, 360 + group_size, group_size)
    group_sums = np.zeros(len(bins)-1)
    for angle, dsi in neurons:
        idx = np.digitize(angle % 360, bins) - 1
        group_sums[idx] += dsi
    max_idx = np.argmax(group_sums)
    angle_center = (bins[max_idx] + bins[max_idx+1]) / 2
    return angle_center

def angle_diff_color(angle1, angle2):
    if angle1 is None or angle2 is None:
        return 0.5
    diff = abs((angle1 - angle2 + 180) % 360 - 180)
    return 0.5 + 0.5 * (diff / 180)

# -----------------------------
# VISUALIZATION
# -----------------------------

def plot_circle_max_angles_combined_original():
    print("📥 Reading CSVs ...")
    data_on_left = read_csv_target_types(CSV_FILES_ON['left'], T4_TYPES)
    data_on_right = read_csv_target_types(CSV_FILES_ON['right'], T4_TYPES)
    data_off_left = read_csv_target_types(CSV_FILES_OFF['left'], T5_TYPES + OFF_EXTRA_TYPES)
    data_off_right = read_csv_target_types(CSV_FILES_OFF['right'], T5_TYPES + OFF_EXTRA_TYPES)

    cmap_on = plt.get_cmap('coolwarm')
    cmap_off = plt.get_cmap('coolwarm')

    # ---- Part A: T4 & T5 Sensors ----
    fig, axes = plt.subplots(1, 8, figsize=(32,4))
    for i, t in enumerate(T4_TYPES + T5_TYPES):
        ax = axes[i]
        neurons_left = data_on_left.get(t, []) if t in T4_TYPES else data_off_left.get(t, [])
        neurons_right = data_on_right.get(t, []) if t in T4_TYPES else data_off_right.get(t, [])

        # Predicted Angle calculation
        angle_left = get_max_dsi_group_angle(neurons_left)
        angle_right = get_max_dsi_group_angle(neurons_right)
        angle = np.mean([a for a in [angle_left, angle_right] if a is not None]) if angle_left or angle_right else None

        circle = plt.Circle((0,0),1,fill=False, lw=20, color='darkgray', alpha=0.5)
        ax.add_patch(circle)

        # Plots Predicted Angle (Model Output)
        if angle is not None:
            theta = np.deg2rad(angle)
            x = np.cos(theta)
            y = np.sin(theta)
            ax.plot(x, y, 'o', markersize=30, color=cmap_on(-0.6), alpha=0.7)

        exp_angle_left = EXPERIMENT_ANGLES.get(t, {}).get('left')
        exp_angle_right = EXPERIMENT_ANGLES.get(t, {}).get('right')
        exp_angle = np.mean([a for a in [exp_angle_left, exp_angle_right] if a is not None])
        if exp_angle is not None:
            theta_exp = np.deg2rad(exp_angle)
            x_exp = np.cos(theta_exp)
            y_exp = np.sin(theta_exp)
            ax.plot(x_exp, y_exp, 'o', markersize=30, color='green', alpha=0.8)

            if angle is not None:
                color_alpha = angle_diff_color(angle, exp_angle)
                ax.fill([0, x, x_exp], [0, y, y_exp], color='yellow', alpha=color_alpha, zorder=0)

        ax.set_xlim(-1.2,1.2)
        ax.set_ylim(-1.2,1.2)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(t, fontsize=20)

    out_path = os.path.join(RESULTS_DIR, f"T4T5_circle.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"✅ Saved T4/T5 combined circle plot -> {out_path}")

    # ---- Part B: High-order OFF Types (HS/VS) ----
    all_off_types = ['HSN','HSE','HSS'] + ['VS1','VS2','VS3','VS4','VS5','VS6','VS7','VS8']
    fig, axes = plt.subplots(1, len(all_off_types), figsize=(44,4))
    for i, t in enumerate(all_off_types):
        ax = axes[i]
        neurons_left = data_off_left.get(t, [])
        neurons_right = data_off_right.get(t, [])

        angle_left = get_max_dsi_group_angle(neurons_left)
        angle_right = get_max_dsi_group_angle(neurons_right)
        angle = np.mean([a for a in [angle_left, angle_right] if a is not None]) if angle_left or angle_right else None

        circle = plt.Circle((0,0),1,fill=False, lw=20, color='darkgray', alpha=0.5)
        ax.add_patch(circle)

        if angle is not None:
            theta = np.deg2rad(angle)
            x = np.cos(theta)
            y = np.sin(theta)
            ax.plot(x, y, 'o', markersize=30, color=cmap_off(0.9), alpha=0.7)

        exp_angle_left = EXPERIMENT_ANGLES.get(t, {}).get('left')
        exp_angle_right = EXPERIMENT_ANGLES.get(t, {}).get('right')
        exp_angle = np.mean([a for a in [exp_angle_left, exp_angle_right] if a is not None])
        if exp_angle is not None:
            theta_exp = np.deg2rad(exp_angle)
            x_exp = np.cos(theta_exp)
            y_exp = np.sin(theta_exp)
            ax.plot(x_exp, y_exp, 'o', markersize=30, color='green', alpha=0.8)

            if angle is not None:
                color_alpha = angle_diff_color(angle, exp_angle)
                ax.fill([0, x, x_exp], [0, y, y_exp], color='yellow', alpha=color_alpha, zorder=0)

        ax.set_xlim(-1.2,1.2)
        ax.set_ylim(-1.2,1.2)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(t, fontsize=20)

    out_path = os.path.join(RESULTS_DIR, f"OFF_circle.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"✅ Saved OFF combined circle plot -> {out_path}")


def main():
    plot_circle_max_angles_combined_original()

if __name__ == '__main__':
    main()

# Hemispheric Tuning Validation: Circular Alignment Atlas
This module validates the directional tuning of neurons across the left and right hemispheres. By comparing predicted Preferred Angles against Biological Ground Truth, it visualizes hemispheric symmetry and directional accuracy through circular plots and angular deviation shading.

**Core Functionalities:**
* **Hemispheric Comparison:** Independently analyzes tuning data for left/right brain regions to verify functional symmetry.

* **Peak Population Inference:** Uses DSI-weighted binning to identify the most robust tuning direction for each cell type.

* **Visual Error Metrics:** Employs circular markers—Blue for Model Prediction, Dark Red for Experimental Benchmark—and shaded sectors to represent the tuning gap.

* **Multi-Pathway Validation:** Covers ON (T4), OFF (T5), and high-order tangential cells (HS/VS).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import csv
import scienceplots

# -----------------------------
# 1. CONFIG & SYSTEM SETUP
# -----------------------------
plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],  
    "font.size": 20,
    "axes.labelsize": 24,
    "axes.titlesize": 40,
    "legend.fontsize": 40,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "axes.linewidth": 1.5,
})

RESULTS_DIR = './results/DSI/circle_max_angle/'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Data source mapping for hemispheric DSI results
CSV_FILES_OFF = {
    'left': './output/DSI/DSI_OFF_left.csv',
    'right': './output/DSI/DSI_OFF_right.csv'
}
CSV_FILES_ON = {
    'left': './output/DSI/DSI_ON_left.csv',
    'right': './output/DSI/DSI_ON_right.csv'
}

# Biological ground truth angles for hemispheric direction selectivity.
T4_TYPES = ['T4a','T4b','T4c','T4d']
T5_TYPES = ['T5a','T5b','T5c','T5d']
OFF_EXTRA_TYPES = ['HSN','HSE','HSS','VS1','VS2','VS3','VS4','VS5','VS6','VS7','VS8']

EXPERIMENT_ANGLES = {
    'T4a': {'left':180,'right':0},
    'T4b': {'left':0,'right':180},
    'T4c': {'left':90,'right':90},
    'T4d': {'left':270,'right':270},
    'T5a': {'left':180,'right':0},
    'T5b': {'left':0,'right':180},
    'T5c': {'left':90,'right':90},
    'T5d': {'left':270,'right':270},
    'VS1': {'left':270,'right':270},
    'VS2': {'left':270,'right':270},
    'VS3': {'left':270,'right':270},
    'VS4': {'left':270,'right':270},
    'VS5': {'left':270,'right':270},
    'VS6': {'left':270,'right':270},
    'VS7': {'left':270,'right':270},
    'VS8': {'left':270,'right':270},
    'HSN': {'left':180,'right':0},
    'HSE': {'left':180,'right':0},
    'HSS': {'left':180,'right':0},
}

# -----------------------------
# 2. DATA UTILITIES
# -----------------------------
def read_csv_target_types(csv_path, target_types):
    data = {t: [] for t in target_types}
    with open(csv_path, 'r', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            t = row['type']
            if t not in target_types:
                continue
            try:
                angle = float(row.get('PreferredAngle', -1))
                if angle < 0:
                    continue
                dsi = float(row['DSI'])
                data[t].append((angle, dsi))
            except:
                continue
    return data

# Calculates the population tuning peak via DSI-weighted angular binning.
def get_max_dsi_group_angle(neurons, group_size=10):
    if len(neurons) == 0:
        return None
    bins = np.arange(0, 360 + group_size, group_size)
    group_sums = np.zeros(len(bins)-1)
    for angle, dsi in neurons:
        idx = np.digitize(angle % 360, bins) - 1
        group_sums[idx] += dsi
    max_idx = np.argmax(group_sums)
    angle_center = (bins[max_idx] + bins[max_idx+1]) / 2
    return angle_center

# Computes intensity/alpha based on circular distance (Angular Error).
def angle_diff_color(angle1, angle2):
    if angle1 is None or angle2 is None:
        return 0.5
    diff = abs((angle1 - angle2 + 180) % 360 - 180)
    return 0.5 + 0.5 * (diff / 180)

# -----------------------------
# 3. HEMISPHERIC ATLAS GENERATION
# -----------------------------
def plot_circle_max_angles_left_right():
    print("📥 Reading CSVs ...")
    data_on_left = read_csv_target_types(CSV_FILES_ON['left'], T4_TYPES)
    data_on_right = read_csv_target_types(CSV_FILES_ON['right'], T4_TYPES)
    data_off_left = read_csv_target_types(CSV_FILES_OFF['left'], T5_TYPES + OFF_EXTRA_TYPES)
    data_off_right = read_csv_target_types(CSV_FILES_OFF['right'], T5_TYPES + OFF_EXTRA_TYPES)

    cmap_on = plt.get_cmap('coolwarm')
    cmap_off = plt.get_cmap('coolwarm')

    # ---- Part A: T4/T5 Sensory Neurons (2-Row Grid: Left vs Right) ----
    all_t4t5 = T4_TYPES + T5_TYPES
    fig, axes = plt.subplots(2, len(all_t4t5), figsize=(32,8))
    for i, t in enumerate(all_t4t5):
        for j, side in enumerate(['left','right']):
            ax = axes[j,i]
            # Select appropriate data pool based on pathway and side
            neurons = (data_on_left if t in T4_TYPES else data_off_left)[t] if side=='left' else (data_on_right if t in T4_TYPES else data_off_right)[t]
            angle = get_max_dsi_group_angle(neurons)
            
            # Drawing the circular reference frame
            circle = plt.Circle((0,0),1,fill=False, lw=20, color='darkgray', alpha=0.5)
            ax.add_patch(circle)

            # Plot Model Prediction (Blue Marker)
            if angle is not None:
                theta = np.deg2rad(angle)
                x = np.cos(theta)
                y = np.sin(theta)
                ax.plot(x, y, 'o', markersize=30, color='#1E3A8A', alpha=0.7)
            # Plot Experimental Benchmark (Dark Red Marker) & Deviation Shading
            exp_angle = EXPERIMENT_ANGLES.get(t, {}).get(side)
            if exp_angle is not None:
                theta_exp = np.deg2rad(exp_angle)
                x_exp = np.cos(theta_exp)
                y_exp = np.sin(theta_exp)
                ax.plot(x_exp, y_exp, 'o', markersize=30, color='#8B0000', alpha=0.7)
                if angle is not None:
                    color_alpha = angle_diff_color(angle, exp_angle)
                    ax.fill([0, x, x_exp], [0, y, y_exp], color='darkgray', alpha=0.7, zorder=0)

            ax.set_xlim(-1.2,1.2)
            ax.set_ylim(-1.2,1.2)
            ax.set_aspect('equal')
            ax.axis('off')
            ax.set_title(f"{t} {side}", fontsize=50)

    out_path = os.path.join(RESULTS_DIR, f"T4T5_circle_left_right.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"✅ Saved T4/T5 left/right circle plot -> {out_path}")

    # ---- Part B: High-order OFF Pathway (HS/VS) ----
    all_off = ['HSN','HSE','HSS'] + ['VS1','VS2','VS3','VS4','VS5','VS6','VS7','VS8']
    fig, axes = plt.subplots(2, len(all_off), figsize=(44,8))
    for i, t in enumerate(all_off):
        for j, side in enumerate(['left','right']):
            ax = axes[j,i]
            neurons = data_off_left[t] if side=='left' else data_off_right[t]
            angle = get_max_dsi_group_angle(neurons)

            circle = plt.Circle((0,0),1,fill=False, lw=20, color='darkgray', alpha=0.5)
            ax.add_patch(circle)

            if angle is not None:
                theta = np.deg2rad(angle)
                x = np.cos(theta)
                y = np.sin(theta)
                ax.plot(x, y, 'o', markersize=30, color='#1E3A8A', alpha=0.7)

            exp_angle = EXPERIMENT_ANGLES.get(t, {}).get(side)
            if exp_angle is not None:
                theta_exp = np.deg2rad(exp_angle)
                x_exp = np.cos(theta_exp)
                y_exp = np.sin(theta_exp)
                ax.plot(x_exp, y_exp, 'o', markersize=30, color='#8B0000', alpha=0.7)
                if angle is not None:
                    ax.fill([0, x, x_exp], [0, y, y_exp], color='darkgray', alpha=0.7, zorder=0)

            ax.set_xlim(-1.2,1.2)
            ax.set_ylim(-1.2,1.2)
            ax.set_aspect('equal')
            ax.axis('off')
            ax.set_title(f"{t} {side}", fontsize=50)

    out_path = os.path.join(RESULTS_DIR, f"OFF_circle_left_right.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"✅ Saved OFF left/right circle plot -> {out_path}")


def main():
    plot_circle_max_angles_left_right()

if __name__ == '__main__':
    main()

# Angular Accuracy Analysis: Deviation Distribution
This module quantifies the **prediction error** by calculating the angular difference between the model's preferred directions and experimental benchmarks. It employs a combination of **Box Plots**, **Kernel Density Estimation (KDE) Jittered Scatter**, and **Bar Charts** to visualize the reliability of directional tuning across all neural subtypes.
**Visualization Features:**
* **Error Quantification:** Measures the absolute circular difference ($0^\circ$ to $180^\circ$) between prediction and ground truth.
* **Distribution Density:** Uses KDE-based jittering in scatter plots to reveal where the majority of neuronal predictions cluster.
* **Pathway Contrast:** Color-coded representation—Dark Red for ON pathway (T4) and Deep Blue for OFF pathway (T5/HS/VS).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import csv
import scienceplots
from scipy.stats import gaussian_kde

# -----------------------------
# GLOBAL CONFIGURATION
# -----------------------------
plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 30,
    "axes.labelsize": 30,
    "axes.titlesize": 30,
    "xtick.labelsize": 30,
    "ytick.labelsize": 30,
    "axes.linewidth": 1.5,
})

RESULTS_DIR = '../results/DSI/plots_optimized/'
os.makedirs(RESULTS_DIR, exist_ok=True)

CSV_FILES_OFF = {
    'left': './output/DSI/DSI_OFF_left.csv',
    'right': './output/DSI/DSI_OFF_right.csv'
}
CSV_FILES_ON = {
    'left': './output/DSI/DSI_ON_left.csv',
    'right': './output/DSI/DSI_ON_right.csv'
}

# Category definitions
T4_TYPES = ['T4a','T4b','T4c','T4d']
T5_TYPES = ['T5a','T5b','T5c','T5d']
HS_TYPES = ['HSN','HSE','HSS']
VS_TYPES = ['VS1','VS2','VS3','VS4','VS5','VS6','VS7','VS8']

ALL_TYPES = T4_TYPES + T5_TYPES + HS_TYPES + VS_TYPES

# Literature-based ground truth for preferred motion directions.
EXPERIMENT_ANGLES = {
    'T4a': {'left':180,'right':0},
    'T4b': {'left':0,'right':180},
    'T4c': {'left':90,'right':90},
    'T4d': {'left':270,'right':270},
    'T5a': {'left':180,'right':0},
    'T5b': {'left':0,'right':180},
    'T5c': {'left':90,'right':90},
    'T5d': {'left':270,'right':270},
    'VS1': {'left':270,'right':270},
    'VS2': {'left':270,'right':270},
    'VS3': {'left':270,'right':270},
    'VS4': {'left':270,'right':270},
    'VS5': {'left':270,'right':270},
    'VS6': {'left':270,'right':270},
    'VS7': {'left':270,'right':270},
    'VS8': {'left':270,'right':270},
    'HSN': {'left':180,'right':0},
    'HSE': {'left':180,'right':0},
    'HSS': {'left':180,'right':0},
}

# -----------------------------
# MATHEMATICAL UTILITIES
# -----------------------------
def read_csv_target_types(csv_path, target_types):
    data = {t: [] for t in target_types}
    with open(csv_path, 'r', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            t = row['type']
            if t not in target_types:
                continue
            try:
                angle = float(row.get('PreferredAngle', -1))
                if angle < 0:
                    continue
                data[t].append(angle % 360)
            except:
                continue
    return data

def angle_difference(a1, a2):
    return abs((a1 - a2 + 180) % 360 - 180)

# Aggregates error values for all subtypes and hemispheres.
def collect_angle_diffs(types_list):
    data_on_left = read_csv_target_types(CSV_FILES_ON['left'], T4_TYPES)
    data_on_right = read_csv_target_types(CSV_FILES_ON['right'], T4_TYPES)
    data_off_left = read_csv_target_types(CSV_FILES_OFF['left'], T5_TYPES + HS_TYPES + VS_TYPES)
    data_off_right = read_csv_target_types(CSV_FILES_OFF['right'], T5_TYPES + HS_TYPES + VS_TYPES)

    type_diffs = {}
    for t in types_list:
        left_exp = EXPERIMENT_ANGLES[t]['left']
        right_exp = EXPERIMENT_ANGLES[t]['right']
        left_angles = data_on_left.get(t, []) if t in T4_TYPES else data_off_left.get(t, [])
        right_angles = data_on_right.get(t, []) if t in T4_TYPES else data_off_right.get(t, [])
        type_diffs[f"{t}-L"] = [angle_difference(a, left_exp) for a in left_angles]
        type_diffs[f"{t}-R"] = [angle_difference(a, right_exp) for a in right_angles]
    return type_diffs

# -----------------------------
# DISTRIBUTION PLOTTING
# -----------------------------
def plot_scatter_box_T4T5(type_diffs):
    types_list = T4_TYPES + T5_TYPES
    fig, ax = plt.subplots(figsize=(16,6))
    
    positions = []
    tick_labels = []

    for i, t in enumerate(types_list):
        for j, side in enumerate(['L','R']):
            key = f"{t}-{side}"
            values = type_diffs[key]
            if not values:
                continue
            pos = i*2 + j + 1
            positions.append(pos)
            tick_labels.append(key)

            # Draw Box Plot (Summary Statistics)
            color = '#8B0000' if t in T4_TYPES else '#1E3A8A'
            bp = ax.boxplot(values, positions=[pos], widths=0.6, patch_artist=True,
                            showfliers=False,
                            boxprops=dict(facecolor=color, alpha=0.3, edgecolor='k', linewidth=2), 
                            medianprops=dict(color='k', linewidth=5), 
                            whiskerprops=dict(color='k', linewidth=2), 
                            capprops=dict(color='k', linewidth=2)) 

            if len(values) > 1:
                kde = gaussian_kde(values)
                y_vals = np.array(values)
                densities = kde(y_vals)
                densities /= densities.max()  # 0-1
                jitter = (np.random.rand(len(values)) - 0.5) * 0.6 * densities
                ax.scatter(np.ones(len(values))*pos + jitter, y_vals, color=color, alpha=0.3, s=15, edgecolors=None)
            else:
                ax.scatter(pos, values[0], color=color, alpha=0.3, s=15, edgecolors=None)

    ax.set_xticks(positions)
    ax.set_xticklabels(tick_labels, rotation=45)
    ax.set_ylabel('Angle Difference (deg)')
    ax.set_ylim(0,200)
    plt.tight_layout()
    out_path = os.path.join(RESULTS_DIR, 'T4T5_scatter_box_density.png')
    plt.savefig(out_path, dpi=200)
    plt.close(fig)
    print(f"✅ Saved T4/T5 scatter+box density plot -> {out_path}")

def plot_bar_HSVS(type_diffs):
    types_list = HS_TYPES + VS_TYPES
    labels = []
    values = []
    colors = []
    nature_colors = ['#1E3A8A','#8B0000'] 
    for t in types_list:
        for side, color in zip(['L','R'], nature_colors):
            key = f"{t}-{side}"
            vals = type_diffs[key]
            if vals:
                for v in vals:
                    values.append(v)
                    labels.append(key)
                    colors.append(color)

    x = np.arange(len(values))
    fig, ax = plt.subplots(figsize=(16,6))
    ax.bar(x, values, color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45)
    ax.set_ylabel('Angle Difference (deg)')
    ax.set_ylim(0,180)
    ax.set_title('HS and VS Subtypes Angle Difference')
    plt.tight_layout()
    out_path = os.path.join(RESULTS_DIR, 'HSVS_bar_nature.png')
    plt.savefig(out_path, dpi=200)
    plt.close(fig)
    print(f"✅ Saved HS/VS bar plot (nature colors) -> {out_path}")

def main():
    type_diffs = collect_angle_diffs(ALL_TYPES)
    plot_scatter_box_T4T5(type_diffs)
    plot_bar_HSVS(type_diffs)

if __name__ == '__main__':
    main()

# Looming Response Atlas: Spatiotemporal Mapping
This script processes large-scale neuronal response data from **Looming Stimuli** (approaching objects). It maps temporal firing traces onto a **2D spatial grid** ($41 \times 82$) to visualize receptive fields and response peaks across the visual space.

**Key Visual Outputs:**

* **Heatmap:** Spatial distribution of peak response magnitudes.

* **Trace Grid:** A dense matrix of temporal waveforms across the visual field.

* **Contour Plot:** Topographical visualization of the neuron's "Looming Sensitivity Field."

* **Peak Trace:** A high-resolution extraction of the strongest temporal response for kinetic analysis.

In [ ]:
import os 
import glob
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from matplotlib.colors import Normalize
from matplotlib.cm import get_cmap
import scienceplots

# -----------------------------
# 1. STYLE & PLOT CONFIG
# -----------------------------
plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 40,
    "axes.labelsize": 40,
    "axes.titlesize": 40,
    "xtick.labelsize": 40,
    "ytick.labelsize": 40,
    "axes.linewidth": 1.5,
})


responses_dir = "./output/loom/202603/"

heat_dir = "../results/loom/plots_heat/"
trace_dir = "../results/loom/plots_trace/"
contour_dir = "../results/loom/plots_contour/"
peak_trace_root = "../results/loom/plots_peak_trace/"

os.makedirs(heat_dir, exist_ok=True)
os.makedirs(trace_dir, exist_ok=True)
os.makedirs(contour_dir, exist_ok=True)
os.makedirs(peak_trace_root, exist_ok=True)

cell_type_file = "../data/cell_type.txt"
classification_file = "../data/classification.txt"

# -----------------------------
# 2. DATA LOADING & FILTERING
# -----------------------------

GRID_Y = 41
GRID_X = 82

TARGET_TYPES = {"DNp01", "DNp02", "DNp03", "DNp04", "DNp11"}

cell_type_df = pd.read_csv(
    cell_type_file,
    header=None,
    names=["root_id", "primary_type", "additional_type"]
)
cell_type_df.set_index("root_id", inplace=True)
cell_type_df.index = cell_type_df.index.astype(str)

classification_df = pd.read_csv(
    classification_file,
    dtype={"root_id": str}
)
classification_df.set_index("root_id", inplace=True)
classification_df.index = classification_df.index.astype(str)

all_files = glob.glob(os.path.join(responses_dir, "*.npz"))
responses_by_ms = {}

print("Loading response files...")
for f in tqdm(all_files):
    base = os.path.basename(f)

    try:
        x = int(base.split("_")[0][1:])
        y = int(base.split("_")[1][1:])
    except Exception:
        continue

    if x >= GRID_X or y >= GRID_Y:
        continue

    ms = base.split("_")[-1].replace(".npz", "")
    data = np.load(f, allow_pickle=True)

    if "neuron_ids" not in data or "responses" not in data:
        continue

    neuron_ids = data["neuron_ids"].astype(str)
    responses = data["responses"]

    stim_dict = responses_by_ms.setdefault(ms, {}).setdefault((x, y), {})
    for nid, resp in zip(neuron_ids, responses):
        stim_dict[nid] = resp


neurons_to_plot_by_ms = {}
for ms, stim_dict in responses_by_ms.items():
    neuron_set = set()
    for neuron_resp_dict in stim_dict.values():
        neuron_set.update(neuron_resp_dict.keys())
    neurons_to_plot_by_ms[ms] = [
        nid for nid in neuron_set
        if cell_type_df["primary_type"].get(nid, "Unknown") in TARGET_TYPES
    ]

for ms, neuron_list in neurons_to_plot_by_ms.items():
    print(f"\nMS: {ms} ( {len(neuron_list)} 个 neuron)")
    for neuron_id in neuron_list:
        neuron_type = cell_type_df["primary_type"].get(neuron_id, "Unknown")
        neuron_side = classification_df["side"].get(neuron_id, "Unknown")
        print(f"Neuron ID: {neuron_id}, Type: {neuron_type}, Side: {neuron_side}")

# -----------------------------
# 3. CORE VISUALIZATION ENGINE
# -----------------------------
def plot_single_neuron(neuron_id, ms, stim_dict):
    # Generates a multi-modal visual profile for a single neuron.
    neuron_type = cell_type_df["primary_type"].get(neuron_id, "Unknown")
    neuron_side = classification_df["side"].get(neuron_id, "Unknown")

    peak_map = np.full((GRID_Y, GRID_X), np.nan, dtype=np.float32)
    response_grid = [[None]*GRID_X for _ in range(GRID_Y)]

    # Map responses to spatial coordinates
    for (x, y), neuron_resp_dict in stim_dict.items():
        if neuron_id not in neuron_resp_dict:
            continue

        resp = neuron_resp_dict[neuron_id]
        response_grid[y][x] = resp
        peak_map[y, x] = max(np.max(resp), 0.0)

    vmax = np.nanmax(peak_map)
    if not np.isfinite(vmax):
        return

    y_peak, x_peak = np.unravel_index(
        np.nanargmax(peak_map),
        peak_map.shape
    )
    peak_resp = response_grid[y_peak][x_peak]

    print(f"Plotting Neuron: {neuron_id}, Type: {neuron_type}, Side: {neuron_side}, Peak Response: {np.max(peak_resp):.3f}")

    fname = f"{neuron_type}_{neuron_side}_{neuron_id}_{ms}.png"

    # =================================================
    # Heatmap
    # =================================================
    cmap_heat = get_cmap("magma").copy()
    cmap_heat.set_bad("white")
    cmap_heat.set_under("black")

    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(
        peak_map,
        origin="lower",
        cmap=cmap_heat,
        norm=Normalize(vmin=0.5*vmax, vmax=vmax),
        aspect="equal"
    )
    ax.axis("off")
    
    suffix = "_R" if neuron_side.lower() == "right" else "_L"
    title_text = f"{neuron_type}{suffix}"
    ax.set_title(title_text, pad=10)
    fig.savefig(os.path.join(heat_dir, fname), dpi=600, bbox_inches="tight")
    plt.close(fig)

    # =================================================
    # Trace Grid
    # =================================================
    fig = plt.figure(figsize=(10, 5))
    gs = fig.add_gridspec(GRID_Y, GRID_X, wspace=0.01, hspace=0.01)

    for y in range(GRID_Y):
        for x in range(GRID_X):
            ax = fig.add_subplot(gs[GRID_Y - 1 - y, x])
            ax.axis("off")

            resp = response_grid[y][x]
            if resp is None:
                continue

            peak = np.max(resp)
            ax.plot(
                resp,
                color="black",
                linewidth=0.5
            )
            ax.set_ylim(-vmax, vmax)

    fig.savefig(os.path.join(trace_dir, fname), dpi=600, bbox_inches="tight")
    plt.close(fig)

    # =================================================
    # Topographical Contour
    # =================================================
    cmap_contour = get_cmap("plasma").copy()
    cmap_contour.set_bad("white")
    cmap_contour.set_under("black")

    levels = np.linspace(0, vmax, 16)
    fig, ax = plt.subplots(figsize=(10, 5))
    cf = ax.contourf(
        peak_map,
        levels=levels,
        cmap=cmap_contour,
        norm=Normalize(vmin=0, vmax=vmax)
    )
    ax.contour(
        peak_map,
        levels=levels,
        colors="k",
        linewidths=0.4,
        alpha=0.4
    )
    ax.axis("off")
    fig.colorbar(cf, ax=ax, fraction=0.046, pad=0.02)
    fig.savefig(os.path.join(contour_dir, fname), dpi=600, bbox_inches="tight")
    plt.close(fig)

    # =================================================
    # Peak Trace Extraction
    # =================================================
    peak_val = np.max(peak_resp)
    color = cmap_heat(peak_val / vmax)

    fig, ax = plt.subplots(figsize=(3.2, 1.6))
    ax.plot(peak_resp, color="black", linewidth=0.5)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.tick_params(left=False, labelleft=False)

    ax.set_ylim(-peak_val * 1.05, peak_val * 1.05)
    ax.set_xlim(0, len(peak_resp) - 1)

    peak_fname = f"{neuron_type}_{neuron_side}_{neuron_id}_{ms}_peak_trace.png"

    fig.savefig(
        os.path.join(peak_trace_root, peak_fname),
        dpi=300,
        bbox_inches="tight"
    )
    plt.close(fig)

    gc.collect()

# -----------------------------
# 4. EXECUTION LOOP
# -----------------------------

print("\nStart plotting...")
for ms, stim_dict in responses_by_ms.items():
    for neuron_id in tqdm(neurons_to_plot_by_ms[ms], desc=f"{ms}"):
        plot_single_neuron(neuron_id, ms, stim_dict)

# Synaptic Symmetry Atlas: Bilateral Weight Distribution
This script generates a **back-to-back horizontal bar chart** to compare the total synaptic input from different **Gustatory Receptor Neurons (GRNs)** onto the left and right MN9 motor neurons. It quantifies hemispheric symmetry by mapping aggregate weights onto a divergent coordinate system.
**Key Visual Outputs:**
* **Bilateral Bar Chart:** Juxtaposes left-side (negative x-axis) and right-side (positive x-axis) inputs for direct comparison.

* **Color-Coded Intensity:** Bars are mapped to a $Coolwarm$ colormap, visually representing the strength of the functional connection.

* **Normalized Scales:** Uses absolute magnitude for axis labels to facilitate symmetry checks across different weight ranges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import scienceplots

# -----------------------------
# CONFIG
# -----------------------------
plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 24,
    "axes.labelsize": 24,
    "axes.titlesize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 24,
    "axes.linewidth": 1.0,
})

# Unique identifiers for the target MN9 neurons (Left/Right)
left_mn9 = 720575940618238523
right_mn9 = 720575940660219265

grn_files = {
    "Ir94e": ["output/Ir94e_left_excitatory.npz", "output/Ir94e_left_inhibitory.npz"],
    "Sugar": ["output/Sugar_left_excitatory.npz", "output/Sugar_left_inhibitory.npz"],
    "Bitter": ["output/Bitter_left_excitatory.npz", "output/Bitter_left_inhibitory.npz"],
    "Phantom": ["output/Phantom_left_excitatory.npz", "output/Phantom_left_inhibitory.npz"]
}

# -----------------------------
# DATA LOADING
# -----------------------------
def load_matrix(file_path):
    data = np.load(file_path, allow_pickle=True)
    return data["target_ids"], data["weight_matrix"]

left_values = []
right_values = []

# Iterate through cell types to sum total synaptic input for MN9
for grn, (exc_file, inh_file) in grn_files.items():

    tgt_exc, exc_matrix = load_matrix(exc_file)
    tgt_inh, inh_matrix = load_matrix(inh_file)
    
    # Sum excitatory and inhibitory weights for the Left MN9
    if left_mn9 in tgt_exc:
        idx_left = np.where(tgt_exc == left_mn9)[0][0]
        total_left = np.sum(exc_matrix[:, idx_left] + inh_matrix[:, idx_left])
    else:
        total_left = 0.0
    
    # Sum excitatory and inhibitory weights for the Right MN9
    if right_mn9 in tgt_exc:
        idx_right = np.where(tgt_exc == right_mn9)[0][0]
        total_right = np.sum(exc_matrix[:, idx_right] + inh_matrix[:, idx_right])
    else:
        total_right = 0.0

    left_values.append(total_left)
    right_values.append(total_right)

# -----------------------------
# VISUALIZATION
# -----------------------------
labels = list(grn_files.keys())
y_pos = np.arange(len(labels))

# Normalize color map based on the global maximum weight absolute value
all_values = np.array(left_values + right_values)
vmax = np.max(np.abs(all_values))
norm = plt.Normalize(-vmax, vmax)
cmap = plt.cm.coolwarm

# Assign specific colors to bars based on their synaptic weight intensity
colors_left = cmap(norm(left_values))
colors_right = cmap(norm(right_values))

fig, ax = plt.subplots(figsize=(10, 6))

bar_height = 0.6

# Plot Left MN9 inputs on the negative x-axis side
ax.barh(
    y_pos,
    -np.abs(left_values),
    color=colors_left,
    height=bar_height
)

# Plot Right MN9 inputs on the positive x-axis side
ax.barh(
    y_pos,
    np.abs(right_values),
    color=colors_right,
    height=bar_height
)


ax.axvline(0, linewidth=1)


ax.grid(axis='x', linestyle='--', alpha=0.3)


ax.set_yticks(y_pos)
ax.set_yticklabels(labels)


ax.set_xlim(-vmax, vmax)


xticks = np.linspace(-vmax, vmax, 5)
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.5f}" for x in xticks])

ax.set_xlabel("Sum of FM")


ax.text(
    -vmax * 0.6,
    len(labels) - 0.3,
    "MN9 Left",
    ha='center',
    va='bottom'
)

ax.text(
    vmax * 0.6,
    len(labels) - 0.3,
    "MN9 Right",
    ha='center',
    va='bottom'
)

# ====== Colorbar ======
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label("FM weight")

plt.tight_layout()
plt.show()

# Synaptic Barcode: Single-Neuron Resolution Mapping
This script generates a **bilaterally symmetrical heatmap matrix** to visualize the specific contribution of individual **Gustatory Receptor Neurons (GRNs)** to the MN9 motor neurons. It decomposes population-level connectivity into single-source resolution for circuit analysis.
**Key Visual Outputs:**
* **Bilateral Barcode:** A mirrored matrix where the central axis represents the MN9 targets.
* **Source-Specific Weights:** Each Rectangle represents a unique source neuron, revealing population heterogeneity.
* **Net Functional Mapping:** Uses a divergent colormap ($Coolwarm$) to represent the integrated effect of excitatory and inhibitory inputs.
* **Symmetry Atlas:** Provides a direct visual comparison of developmental stereotypy between the left and right hemispheres.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib as mpl
import scienceplots

# -----------------------------
# CONFIG
# -----------------------------
plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 24,
    "axes.labelsize": 24,
    "axes.titlesize": 24,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "axes.linewidth": 1.0,
})

# IDs for target MN9 motor neurons
mn9_left = 720575940618238523
mn9_right = 720575940660219265

grn_files = {
    "Ir94e": ["output/Ir94e_left_excitatory.npz", "output/Ir94e_left_inhibitory.npz"],
    "Sugar": ["output/Sugar_left_excitatory.npz", "output/Sugar_left_inhibitory.npz"],
    "Bitter": ["output/Bitter_left_excitatory.npz", "output/Bitter_left_inhibitory.npz"],
    "Phantom": ["output/Phantom_left_excitatory.npz", "output/Phantom_left_inhibitory.npz"]
}

# -----------------------------
# DATA PROCESSING
# -----------------------------
def load_matrix(file_path):
    data = np.load(file_path, allow_pickle=True)
    return data["source_ids"], data["target_ids"], data["weight_matrix"]

# Aggregate synaptic weight vectors for specified left and right targets
def collect_grn_vectors(target_left, target_right):

    all_values = []
    vectors = {}

    for grn, (exc_file, inh_file) in grn_files.items():

        src_exc, tgt_exc, exc_matrix = load_matrix(exc_file)
        src_inh, tgt_inh, inh_matrix = load_matrix(inh_file)
        
        # Extract combined weight vector for the left target
        if target_left in tgt_exc:
            idx_left = np.where(tgt_exc == target_left)[0][0]
            vec_left = exc_matrix[:, idx_left] + inh_matrix[:, idx_left]
        else:
            vec_left = np.zeros(len(src_exc))

        # Extract combined weight vector for the right target
        if target_right in tgt_exc:
            idx_right = np.where(tgt_exc == target_right)[0][0]
            vec_right = exc_matrix[:, idx_right] + inh_matrix[:, idx_right]
        else:
            vec_right = np.zeros(len(src_exc))

        vectors[grn] = (vec_left, vec_right)

        all_values.extend(vec_left)
        all_values.extend(vec_right)

    return vectors, all_values

# -----------------------------
# SYMMETRICAL MATRIX PLOTTING
# -----------------------------
mn9_vectors, mn9_all = collect_grn_vectors(mn9_left, mn9_right)

vmax = np.max(np.abs(mn9_all))
norm = plt.Normalize(-vmax, vmax)
cmap = plt.cm.coolwarm

fig, ax = plt.subplots(figsize=(16, 6))

square_size = 0.8
gap = 0.2

y_positions = np.arange(len(mn9_vectors))

# Render each synaptic weight as a colored rectangle in a symmetrical layout
for row_idx, (grn, (vec_left, vec_right)) in enumerate(mn9_vectors.items()):

    N = len(vec_left)
    
    # Plot left-side neurons extending to the left from the center
    for i in range(N):

        x = - (i + 1) * (square_size + gap)

        color = cmap(norm(vec_left[i]))

        ax.add_patch(
            Rectangle(
                (x, row_idx - square_size/2),
                square_size,
                square_size,
                color=color
            )
        )

    # Plot right-side neurons extending to the right from the center
    for i in range(N):

        x = i * (square_size + gap)

        color = cmap(norm(vec_right[i]))

        ax.add_patch(
            Rectangle(
                (x, row_idx - square_size/2),
                square_size,
                square_size,
                color=color
            )
        )

# Finalize axis and schematic line formatting
ax.axvline(0, color='black', linewidth=1)

ax.set_yticks(y_positions)
ax.set_yticklabels(list(mn9_vectors.keys()))

ax.set_xticks([])

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)

# Calculate spatial extent for centering and labels
max_left = max(len(v[0]) for v in mn9_vectors.values())
max_right = max(len(v[1]) for v in mn9_vectors.values())

max_extent = max(max_left, max_right) * (square_size + gap)

ax.set_xlim(-max_extent - 1, max_extent + 1)
ax.set_ylim(-1, len(mn9_vectors))

ax.text(-max_extent, len(mn9_vectors) + 0.3, 'Left', ha='center')
ax.text(max_extent, len(mn9_vectors) + 0.3, 'Right', ha='center')

ax.text(
    0,
    len(mn9_vectors) + 0.3,
    'MN9',
    ha='center',
    va='center',
    fontweight='bold'
)

# Colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    fraction=0.03,
    pad=0.04
)

cbar.set_label("FM weight")


plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import scienceplots
from matplotlib.ticker import ScalarFormatter


plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 24,
    "axes.labelsize": 24,
    "axes.titlesize": 24,
    "legend.fontsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "axes.linewidth": 1.5,
})

def exp_func(x, a, b):
    return a * np.exp(-b * x)


def plot_weights_vs_depth_box(csv_path, side, save_path=None):

    df = pd.read_csv(csv_path)

    # remove unreachable neurons
    df = df[df["min_depth"] >= 0]

    depths = np.sort(df["min_depth"].unique())

    weight_types = [
        "max_FM_weight",
        "max_postsynaptic_weight",
        "max_presynaptic_weight"
    ]

    colors = ["tab:blue", "tab:orange", "tab:green"]

    titles = [
        "Max FM Weight",
        "Max Postsynaptic Weight",
        "Max Presynaptic Weight"
    ]

    fig, axes = plt.subplots(1, 4, figsize=(22,5), sharey=False)

    ax = axes[0]

    counts = df["min_depth"].value_counts().sort_index()
    depths_count = counts.index.values
    neuron_counts = counts.values

    bars = ax.bar(
        depths_count,
        neuron_counts,
        color="tab:gray",
        alpha=0.7,
        width=0.6
    )

    ax.plot(
        depths_count,
        neuron_counts,
        color="black",
        marker="o",
        linewidth=2
    )

    ax.set_xlabel("Min Propagation Depth")
    ax.set_ylabel(f"Neuron Count")
    ax.set_title("Neuron Distribution")
    ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    ax.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    ax.set_xticks(np.arange(int(df["min_depth"].min()), int(df["min_depth"].max())+1))


    for ax, wtype, color, title in zip(axes[1:], weight_types, colors, titles):

        data_by_depth = [df[df["min_depth"] == d][wtype].values for d in depths]
        means = np.array([np.mean(v) for v in data_by_depth])

        boxprops = dict(facecolor=color, alpha=0.6, linewidth=1.5)
        medianprops = dict(color='black', linewidth=1.5)
        whiskerprops = dict(color=color, linewidth=1.2, linestyle='--')
        capprops = dict(color=color, linewidth=1.2)

        ax.boxplot(
            data_by_depth,
            positions=depths,
            widths=0.6,
            patch_artist=True,
            showfliers=False,
            boxprops=boxprops,
            medianprops=medianprops,
            whiskerprops=whiskerprops,
            capprops=capprops
        )

        # jitter scatter
        for i, vals in enumerate(data_by_depth):
            x = np.random.normal(depths[i], 0.08, size=len(vals))
            ax.scatter(
                x,
                vals,
                color='black',
                alpha=0.4,
                s=8
            )

        # mean line
        ax.plot(
            depths,
            means,
            marker='o',
            linewidth=2,
            color='black',
            label='mean' if i == 1 else None
        )

        if wtype == "max_FM_weight":

            try:
                popt, _ = curve_fit(
                    exp_func,
                    depths,
                    means,
                    p0=(max(means), 0.1)
                )

                a_fit, b_fit = popt

                x_fit = np.linspace(min(depths), max(depths), 200)
                y_fit = exp_func(x_fit, a_fit, b_fit)

                ax.plot(
                    x_fit,
                    y_fit,
                    color='red',
                    linestyle='--',
                    linewidth=2,
                    label=f"{a_fit:.2f}·exp(-{b_fit:.2f}x)"
                )

                print(f"Fit {wtype}: a={a_fit:.3f}, b={b_fit:.3f}")

            except Exception as e:
                print(f"Fit failed for {wtype}: {e}")

        ax.set_xlabel("Min Propagation Depth")
        ax.set_title(title)
        ax.set_xlim(min(depths)-0.5, max(depths)+0.5)

        if ax == axes[1]:
            ax.set_ylabel("Weight")

        ax.legend(loc='best',frameon=False)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)

    plt.show()


plot_weights_vs_depth_box(
    "./output/visual_right_weight_summary.csv",
    side="Right",
    save_path="../results/right_weights_vs_depth_box.png"
)
plot_weights_vs_depth_box(
    "./output/visual_left_weight_summary.csv",
    side="Left",
    save_path="../results/left_weights_vs_depth_box.png"
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import scienceplots
from matplotlib.ticker import ScalarFormatter

plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 28,
    "axes.labelsize": 28,
    "axes.titlesize": 28,
    "legend.fontsize": 28,
    "xtick.labelsize": 28,
    "ytick.labelsize": 28,
    "axes.linewidth": 1.5,
})

def exp_func(x, a, b):
    return a * np.exp(-b * x)

def plot_Strengths_4_figs(left_csv, right_csv, save_path_prefix=None):
    # 读取数据
    df_left = pd.read_csv(left_csv)
    df_right = pd.read_csv(right_csv)
    df_left = df_left[df_left["min_depth"] >= 0]
    df_right = df_right[df_right["min_depth"] >= 0]

    depths_left = np.sort(df_left["min_depth"].unique())
    depths_right = np.sort(df_right["min_depth"].unique())

    # ---------------- 图1：count ----------------
    fig, axes = plt.subplots(1,2, figsize=(16,5), sharey=False)
    for ax, df, depths, side in zip(axes, [df_left, df_right], [depths_left, depths_right], ["Left","Right"]):
        counts = df["min_depth"].value_counts().sort_index()
        ax.bar(counts.index, counts.values, color="tab:gray", alpha=0.7, width=0.6)
        # 不画折线，只保留柱状图
        ax.set_xlabel("Depth")
        if ax==axes[0]:
            ax.set_ylabel("Count")
        # 设置 x 轴刻度为完整 depth
        ax.set_xticks(counts.index)
        ax.set_xticklabels(counts.index, rotation=0)  # 可以改 rotation 调整角度
        ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
        ax.ticklabel_format(style='sci', axis='y', scilimits=(0,0))

    plt.tight_layout()
    if save_path_prefix:
        plt.savefig(f"{save_path_prefix}_count.png", dpi=300)

    # ---------------- 图2：FM ----------------
    fig, axes = plt.subplots(1,2, figsize=(16,5), sharey=False)
    for ax, df, depths, side in zip(axes, [df_left, df_right], [depths_left, depths_right], ["Left","Right"]):
        data_by_depth = [df[df["min_depth"]==d]["max_FM_weight"].values for d in depths]
        means = np.array([np.mean(v) for v in data_by_depth])
        
        ax.boxplot(
            data_by_depth, 
            positions=depths, 
            widths=0.6, 
            patch_artist=True,
            boxprops=dict(facecolor="tab:blue", alpha=0.6), 
            showfliers=False,
            medianprops=dict(color='black', linewidth=1.5),
            whiskerprops=dict(color='tab:blue', linestyle='--'),
            capprops=dict(color='tab:blue')
        )
        
        ax.plot(depths, means, color='black', linewidth=2, label='mean')
        
        try:
            popt, _ = curve_fit(exp_func, depths, means, p0=(max(means),0.1))
            x_fit = np.linspace(min(depths), max(depths), 200)
            y_fit = exp_func(x_fit, *popt)
            ax.plot(x_fit, y_fit, 'r--', linewidth=2, label='exp fit')
        except: 
            pass
        
        ax.set_xlabel("Depth")
        if ax==axes[0]:
            ax.set_ylabel("Max FM Strength")
        
        ax.legend()  # <-- 显示图例

    plt.tight_layout()
    if save_path_prefix:
        plt.savefig(f"{save_path_prefix}_FM.png", dpi=300)

    # ---------------- 图3：post ----------------
    fig, axes = plt.subplots(1,2, figsize=(16,5), sharey=False)
    for ax, df, depths, side in zip(axes, [df_left, df_right], [depths_left, depths_right], ["Left","Right"]):
        data_by_depth = [df[df["min_depth"]==d]["max_postsynaptic_weight"].values for d in depths]
        ax.boxplot(data_by_depth, positions=depths, widths=0.6, patch_artist=True,
                   boxprops=dict(facecolor="tab:orange", alpha=0.6), showfliers=False,
                   medianprops=dict(color='black', linewidth=1.5),
                   whiskerprops=dict(color="tab:orange", linestyle='--'),
                   capprops=dict(color="tab:orange"))
        ax.set_xlabel("Depth")
        if ax==axes[0]:
            ax.set_ylabel(f"Max Post Strength")
    plt.tight_layout()
    if save_path_prefix:
        plt.savefig(f"{save_path_prefix}_post.png", dpi=300)

    # ---------------- 图4：pre ----------------
    fig, axes = plt.subplots(1,2, figsize=(16,5), sharey=False)
    for ax, df, depths, side in zip(axes, [df_left, df_right], [depths_left, depths_right], ["Left","Right"]):
        data_by_depth = [df[df["min_depth"]==d]["max_presynaptic_weight"].values for d in depths]
        ax.boxplot(data_by_depth, positions=depths, widths=0.6, patch_artist=True,
                   boxprops=dict(facecolor="tab:green", alpha=0.6), showfliers=False,
                   medianprops=dict(color='black', linewidth=1.5),
                   whiskerprops=dict(color="tab:green", linestyle='--'),
                   capprops=dict(color="tab:green"))
        ax.set_xlabel("Depth")
        if ax==axes[0]:
            ax.set_ylabel(f"Max Pre Strength")
    plt.tight_layout()
    if save_path_prefix:
        plt.savefig(f"{save_path_prefix}_pre.png", dpi=300)

    plt.show()


# 使用示例
plot_Strengths_4_figs(
    left_csv="./output/visual_left_weight_summary.csv",
    right_csv="./output/visual_right_weight_summary.csv",
    save_path_prefix="../results/combined"
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scienceplots

plt.style.use(['science','nature','no-latex'])
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 28,
    "axes.labelsize": 28,
    "axes.titlesize": 28,
    "legend.fontsize": 28,
    "xtick.labelsize": 28,
    "ytick.labelsize": 28,
    "axes.linewidth": 1.5,
})

depths = [0,1,2,3,4]

# 更漂亮的颜色
color_map = {
    "left":"#1E3A8A",   # 蓝色
    "right":"#8B0000"   # 红色
}

def prepare_merged(csv_path, column_file):
    column_df = pd.read_csv(column_file, sep=",")
    df = pd.read_csv(csv_path)
    merged = column_df.merge(
        df[['root_id','min_depth']],
        on='root_id',
        how='inner'
    )
    merged = merged[merged['min_depth'].between(0,4)]
    return merged

def compute_stats(merged, hemi_filter):
    stats = (
        merged[merged['hemisphere']==hemi_filter]
        .groupby(['type','hemisphere','min_depth'])
        .size()
        .reset_index(name="count")
    )
    totals = (
        merged[merged['hemisphere']==hemi_filter]
        .groupby(['type','hemisphere'])
        .size()
        .reset_index(name="total")
    )
    stats = stats.merge(totals,on=['type','hemisphere'])
    stats["ratio"] = stats["count"] / stats["total"]
    return stats

def plot_mirror_bubble(left_csv, right_csv, column_file, save_path=None):

    left_merged = prepare_merged(left_csv,column_file)
    right_merged = prepare_merged(right_csv,column_file)

    # 只保留同侧
    left_stats = compute_stats(left_merged,'left')
    right_stats = compute_stats(right_merged,'right')

    types = np.sort(np.unique(np.concatenate([left_merged['type'].unique(), right_merged['type'].unique()])))
    types = types[::-1]

    y_pos = np.arange(len(types))
    fig, ax = plt.subplots(figsize=(12,len(types)*0.5))

    max_ratio = max(left_stats["ratio"].max(), right_stats["ratio"].max())
    size_scale = 800 / max_ratio  # 科学比例

    # 左脑左侧
    for i,t in enumerate(types):
        y = y_pos[i]
        lsub = left_stats[left_stats["type"]==t]
        for _,row in lsub.iterrows():
            depth = row["min_depth"]
            ratio = row["ratio"]
            hemi = row["hemisphere"]
            x = -depth
            ax.scatter(
                x,y,
                s=ratio*size_scale,
                color=color_map[hemi],
                alpha=0.3 + 0.6*ratio,
                edgecolor="black",
                linewidth=0.5
            )

    # 右脑右侧
    for i,t in enumerate(types):
        y = y_pos[i]
        rsub = right_stats[right_stats["type"]==t]
        for _,row in rsub.iterrows():
            depth = row["min_depth"]
            ratio = row["ratio"]
            hemi = row["hemisphere"]
            x = depth
            ax.scatter(
                x,y,
                s=ratio*size_scale,
                color=color_map[hemi],
                alpha=0.3 + 0.6*ratio,
                edgecolor="black",
                linewidth=0.5
            )

    # 中轴
    ax.axvline(0,color="black",linewidth=1)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(types)

    xticks = [-4,-3,-2,-1,0,1,2,3,4]
    xticklabels = [4,3,2,1,0,1,2,3,4]
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels)
    ax.set_xlabel("Depth (Left ← → Right)")

    # 图例
    from matplotlib.lines import Line2D
    legend_elements = [
    ]
    # 大小图例
    for s in [0.1,0.3,0.6]:
        legend_elements.append(Line2D([0],[0],marker='o',color='w',label=f'{int(s*100)}% proportion',markerfacecolor='gray',markersize=np.sqrt(s*size_scale),alpha=0.6))

    ax.legend(handles=legend_elements,loc='center left', title='Proportion of Neuron Counts', bbox_to_anchor=(1,0.5),frameon=True,fontsize=22,title_fontsize=20)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path,dpi=600,bbox_inches="tight")
    plt.show()

# 调用
plot_mirror_bubble(
    "./output/visual_left_weight_summary.csv",
    "./output/visual_right_weight_summary.csv",
    "../data/column_assignment.txt",
    save_path="mirror_depth_bubble_same_side.png"
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np
import scienceplots

plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 24,
    "axes.labelsize": 24,
    "axes.titlesize": 24,
    "legend.fontsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "axes.linewidth": 1.5,
})


def plot_connectivity_stats_vs_depth(csv_path, side, save_path=None):
    df = pd.read_csv(csv_path)
    df = df[df["min_depth"] >= 0]

    depths = np.sort(df["min_depth"].unique())

    metrics = [
        "mean_upstream_weight",
        "count_upstream_nonzero",
        "mean_downstream_weight",
        "count_downstream_nonzero"
    ]

    titles = [
        "Mean Upstream Weight",
        "Upstream Connection Count",
        "Mean Downstream Weight",
        "Downstream Connection Count"
    ]

    colors = [
        "tab:blue",
        "tab:orange",
        "tab:green",
        "tab:red"
    ]

    fig, axes = plt.subplots(1, 4, figsize=(24, 5))

    for ax, metric, color, title in zip(axes, metrics, colors, titles):

        data_by_depth = []

        for d in depths:
            vals = df[df["min_depth"] == d][metric].values
            data_by_depth.append(vals)

        means = np.array([np.mean(v) if len(v) > 0 else np.nan for v in data_by_depth])

        boxprops = dict(facecolor=color, alpha=0.6, linewidth=1.5)
        medianprops = dict(color='black', linewidth=1.5)
        whiskerprops = dict(color=color, linewidth=1.2, linestyle='--')
        capprops = dict(color=color, linewidth=1.2)

        ax.boxplot(
            data_by_depth,
            positions=depths,
            widths=0.6,
            patch_artist=True,
            showfliers=False,
            boxprops=boxprops,
            medianprops=medianprops,
            whiskerprops=whiskerprops,
            capprops=capprops
        )

        for i, vals in enumerate(data_by_depth):
            if len(vals) == 0:
                continue
            x = np.random.normal(depths[i], 0.08, size=len(vals))
            ax.scatter(x, vals, color='black', alpha=0.3, s=20)

        ax.plot(depths, means, marker="o", linewidth=2, color="red", label="Mean")

        ax.set_xlabel("Min Propagation Depth")
        ax.set_title(title)
        ax.set_xlim(min(depths) - 0.5, max(depths) + 0.5)

        if "count" in metric:
            ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1e'))

        if ax == axes[0]:
            ax.set_ylabel(f"{side}")

        ax.legend(loc="upper right", frameon=False)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)

    plt.show()


plot_connectivity_stats_vs_depth(
    "./output/visual_right_weight_summary.csv",
    side="Right",
    save_path="../results/right_connectivity_vs_depth_boxplot_scatter.png"
)

plot_connectivity_stats_vs_depth(
    "./output/visual_left_weight_summary.csv",
    side="Left",
    save_path="../results/left_connectivity_vs_depth_boxplot_scatter.png"
)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scienceplots
from matplotlib.colors import LogNorm

plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 18,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "legend.fontsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "axes.linewidth": 1.5,
})


def plot_weight_pairwise_density_by_depth(csv_path, save_path=None, side=''):

    df = pd.read_csv(csv_path)

    df = df[df["min_depth"] >= 0]
    df = df[df["min_depth"] <= 5]

    depths = sorted(df["min_depth"].unique())

    weight_pairs = [
        ("max_FM_weight", "max_postsynaptic_weight"),
        ("max_FM_weight", "max_presynaptic_weight"),
        ("max_postsynaptic_weight", "max_presynaptic_weight")
    ]

    titles = [
        "FM vs Postsynaptic",
        "FM vs Presynaptic",
        "Postsynaptic vs Presynaptic"
    ]

    n_depth = len(depths)

    # -------- log transform --------
    for col in ["max_FM_weight", "max_postsynaptic_weight", "max_presynaptic_weight"]:
        df[col] = np.log10(df[col] + 1e-12)

    # -------- remove outliers via quantile --------
    x_vals = []
    y_vals = []

    for xw, yw in weight_pairs:
        x_vals.extend(df[xw].values)
        y_vals.extend(df[yw].values)

    xmin, xmax = np.percentile(x_vals, [1, 99])
    ymin, ymax = np.percentile(y_vals, [1, 99])

    # -------- figure --------
    fig, axes = plt.subplots(
        3,
        n_depth,
        figsize=(4*n_depth, 11),
        sharex=True,
        sharey=True
    )
    fig.suptitle(f"Neurons")
    hist_ref = None

    for col, depth in enumerate(depths):

        df_d = df[df["min_depth"] == depth]

        for row, ((xw, yw), title) in enumerate(zip(weight_pairs, titles)):

            ax = axes[row, col]

            x = df_d[xw].values
            y = df_d[yw].values

            h = ax.hist2d(
                x,
                y,
                bins=80,
                cmap="coolwarm",
                norm=LogNorm(),
                range=[[xmin, xmax], [ymin, ymax]]
            )

            if hist_ref is None:
                hist_ref = h

            ax.set_xlim(xmin, xmax)
            ax.set_ylim(ymin, ymax)

            ax.set_xlabel(xw.replace("_", " ").title())
            ax.set_ylabel(yw.replace("_", " ").title())

            if col == 0:
                ax.set_title(title)

        axes[0, col].text(
            0.5,
            1.22,
            f"Depth {depth} (n={len(df_d)})",
            transform=axes[0, col].transAxes,
            ha="center",
            va="center"
        )

    # ---------- unified colorbar on far right ----------
    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])  
    # [left, bottom, width, height]

    cbar = fig.colorbar(
        hist_ref[3],
        cax=cbar_ax
    )

    cbar.set_label("Log Density")


    plt.tight_layout(rect=[0, 0, 0.9, 1])

    if save_path:
        plt.savefig(save_path, dpi=300)

    plt.show()


plot_weight_pairwise_density_by_depth(
    "./output/visual_right_weight_summary.csv",
    save_path="../results/Right_weights_pairwise_by_depth.png",
    side='Right'
)
plot_weight_pairwise_density_by_depth(
    "./output/visual_left_weight_summary.csv",
    save_path="../results/Left_weights_pairwise_by_depth.png",
    side='Left'
)

In [ ]:
import navis
import flybrains
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scienceplots
from pathlib import Path
from tqdm import tqdm

plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 20,
    "axes.labelsize": 20,
    "axes.titlesize": 20,
    "axes.linewidth": 1.5,
})

swc_path = Path("../data/sk_lod1_783_healed")
csv_path = "./output/visual_right_weight_summary.csv"

df = pd.read_csv(csv_path)
df = df[(df["min_depth"] >= 0) & (df["min_depth"] <= 5)]
depths = sorted(df["min_depth"].unique())

# -----------------------------
# flywire brain mesh
# -----------------------------
brain_mesh = flybrains.FLYWIRE.mesh

depth_colors = [
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2"
]

print("Scanning SWC directory...")

swc_files = list(swc_path.glob("*.swc"))
id_to_file = {}

for f in swc_files:
    try:
        nid = int(f.stem.split('.')[0])
        id_to_file[nid] = f
    except ValueError:
        continue

print(f"Found {len(id_to_file)} SWC files")

fig, axes = plt.subplots(1, len(depths), figsize=(22,6))


for i, depth in enumerate(tqdm(depths, desc="Plotting depths")):

    fig, ax = plt.subplots(figsize=(6,6))

    df_d = df[df["min_depth"] == depth]

    ids = df_d["root_id"].values
    weights = df_d["max_FM_weight"].values

    if len(ids) == 0:
        continue

    wmin, wmax = weights.min(), weights.max()

    if wmax - wmin < 1e-12:
        norm_w = np.ones_like(weights)
    else:
        norm_w = (weights - wmin) / (wmax - wmin)

    files_to_load = []

    for nid in ids:
        if nid in id_to_file:
            files_to_load.append(id_to_file[nid])

    if not files_to_load:
        continue


    skeletons = navis.read_swc(files_to_load, parallel=True)

    id_to_skeleton = {}

    for sk in skeletons:
        try:
            id_to_skeleton[int(sk.id)] = sk
        except:
            continue
    skels = []
    alphas = []

    for j, nid in enumerate(ids):

        if nid in id_to_skeleton:

            skels.append(id_to_skeleton[nid])

            alpha = 0.15 + 0.85 * norm_w[j]
            alphas.append(alpha)

    navis.plot2d(
        brain_mesh,
        view=("x", "-y", "-z"),
        ax=ax,
        color="lightgrey",
        alpha=0.3
    )

    if skels:

        navis.plot2d(
            skels,
            view=("x", "-y", "-z"),
            ax=ax,
            color=depth_colors[i],
            alpha=alphas,
            linewidth=0.3
        )

    ax.set_title(
        f"Neurons at depth {depth}\n(n = {len(skels)})",
        fontsize=36,color='black'
    )

    ax.set_axis_off()
    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.tight_layout()

    plt.savefig(
        f"right_flywire_depth_{depth}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()